## Imports

In [1]:
import os
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
local_dataset_path = '/Users/E2074/analytics/customer/loyalty/experiment1/analysis'

## Datasets & Parameter

In [5]:
## Parameter 
start_date = '20250412'
end_date = '20250419'
city = 'Bangalore'

### Ad Viewable Impression

In [6]:
## Ad_Viewable_Impression

query_viewable_impression_cux = f"""

select 
    distinct 
    yyyymmdd,
    userid,
    slotname
from 
    canonical.iceberg_log_telemetry_ads_impressions_immutable
where 
    yyyymmdd >= '{start_date}'
    and yyyymmdd <= '{end_date}'
    and city = '{city}'
    and responsetype = 'ONLINE_SALES'
    and eventname = 'Ad_Viewable_Impression'
    and metadata1 like '%Loyalty_MaxBanner%'
"""

# df_viewable_impression_cux = pd.read_sql(query_viewable_impression_cux, connection)

In [7]:
# df_viewable_impression_cux.to_parquet(local_dataset_path + '/customer_viewable_impression.parquet', index=False)
df_viewable_impression_cux = pd.read_parquet(local_dataset_path + '/customer_viewable_impression.parquet')
print(df_viewable_impression_cux.shape)
df_viewable_impression_cux.head()

(509608, 3)


,yyyymmdd,userid,slotname
0,20250415,63882d5a08087f9f39a201fe,CaptainSearch1
1,20250415,65e6d1faa908e321e16789ef,PostOrderStarted1
2,20250416,5d5518fabde10e0d543f8810,CaptainSearch1
3,20250416,64dd1ac3143ba38afdea1091,CaptainSearch1
4,20250416,666a6af10fc0192d5e6095e0,CaptainSearch1


In [8]:
df_viewed_cux = df_viewable_impression_cux[['userid']].drop_duplicates()

In [9]:
print(df_viewable_impression_cux.shape)
print(df_viewed_cux.shape)

(509608, 3)
(135718, 1)


### Orders

In [10]:
## Orders

query_orders = f"""

with experiment_group as (

    select
        -- selectorid,
        case 
        when selectorid in ( '67fe2da559c3280001e34dcd', '67fe2ea4e6819c00019f8b2a', '67fe314d0d2936000122d5de') then 'control'
        when selectorid in ('67f3bb8a5558e600012f7be1', '67f3bcb35f7e1e0001f10470', '67f3bea382692600018fe1db') then 'test'
        end group_tc,
        userid customer_id
    from    
        canonical.user_selector
    where  
        yyyymmdd >= '20250410' 
        and yyyymmdd <= '20250418'
        and selectorid IN ( '67fe2da559c3280001e34dcd', '67fe2ea4e6819c00019f8b2a', '67fe314d0d2936000122d5de', '67f3bb8a5558e600012f7be1', '67f3bcb35f7e1e0001f10470', '67f3bea382692600018fe1db')
    ),
    
    orders as (

    select
        yyyymmdd,
        customer_id,
        service_obj_service_name service_name,
        order_id,
        modified_order_status,
        customer_cancelled_epoch,
        order_requested_epoch,
        accepted_epoch,
        accept_to_cancelled,
        case when modified_order_status = 'COBRA' then ((customer_cancelled_epoch - order_requested_epoch)/1000) end cobra_ttc,
        case when modified_order_status = 'OCARA' then ((customer_cancelled_epoch - accepted_epoch)/1000) end ocara_ttc,
        case when modified_order_status = 'OCARA' then ((accepted_epoch - accepted_epoch)/1000) end tta,
        epoch,
        quarter_hour,
        hhmmss,
        distance_final_distance,
        accept_to_pickup_distance
    from 
        orders.order_logs_fact a
    where 
        yyyymmdd >= '20250405'
        and yyyymmdd <= '20250426'
        and city_name = '{city}'
        and service_obj_service_name in ('Auto','Link','Bike Lite','CabEconomy', 'CabPremium')
    )

    select 
        a.group_tc,
        b.*
    from 
        experiment_group a
    join 
        orders b
        on a.customer_id = b.customer_id
"""

# df_orders = pd.read_sql(query_orders, connection)

In [11]:
# df_orders.to_parquet(local_dataset_path + '/experimentation_orders.parquet', index=False)
df_orders = pd.read_parquet(local_dataset_path + '/experimentation_orders.parquet')
print(df_orders.shape)
df_orders.head()

(3722955, 18)


,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance
0,test,20250413,5a967a8515e10f72cdf433a2,Bike Lite,67fbbb074c6f591c05e18197,dropped,NaN,1744550663081,1.744551e+12,NaN,NaN,NaN,NaN,1744550663081,1845,185423,4.63,1.072
1,test,20250407,5a967a8515e10f72cdf433a2,Link,67f3dab1e27cda2a45500279,COBRA,1.744035e+12,1744034481844,NaN,NaN,49.0,NaN,NaN,1744034481844,1930,193121,4.64,NaN
2,test,20250414,5a967a8515e10f72cdf433a2,Link,67fcffa1b9312e7574368507,OCARA,1.744634e+12,1744633761842,1.744634e+12,0.001462,NaN,199.0,0.0,1744633761842,1745,175921,4.78,2.206
3,test,20250426,5a967a8515e10f72cdf433a2,Link,680c2f666fd2d8338d2f0c99,dropped,NaN,1745629030054,1.745629e+12,NaN,NaN,NaN,NaN,1745629030054,0615,062710,5.04,0.887
4,test,20250425,5a967a8515e10f72cdf433a2,Auto,680b7ceccf29b17087c03c2e,dropped,NaN,1745583340349,1.745583e+12,NaN,NaN,NaN,NaN,1745583340349,1745,174540,5.32,0.676


### Segment

In [12]:
## Orders

query_segment = f"""

with experiment_group as (

    select
        -- selectorid,
        case 
        when selectorid in ( '67fe2da559c3280001e34dcd', '67fe2ea4e6819c00019f8b2a', '67fe314d0d2936000122d5de') then 'control'
        when selectorid in ('67f3bb8a5558e600012f7be1', '67f3bcb35f7e1e0001f10470', '67f3bea382692600018fe1db') then 'test'
        end group_tc,
        userid customer_id
    from    
        canonical.user_selector
    where  
        yyyymmdd >= '20250410' 
        and yyyymmdd <= '20250418'
        and selectorid IN ( '67fe2da559c3280001e34dcd', '67fe2ea4e6819c00019f8b2a', '67fe314d0d2936000122d5de', '67f3bb8a5558e600012f7be1', '67f3bcb35f7e1e0001f10470', '67f3bea382692600018fe1db')
    ),
    
    segment as (

    select 
        customer_id,
        taxi_regularity_segment,
        taxi_service_affinity,
        taxi_retention_segments taxi_retention_segment
    from 
        datasets.iallocator_customer_segments
    where 
        run_date = '2025-04-11'
    )

    select 
        a.group_tc,
        b.*
    from 
        experiment_group a
    join 
        segment b
        on a.customer_id = b.customer_id
"""

# df_segment = pd.read_sql(query_segment, connection)

In [13]:
# df_segment.to_parquet(local_dataset_path + '/customer_segments.parquet', index=False)
df_segment = pd.read_parquet(local_dataset_path + '/customer_segments.parquet')
print(df_segment.shape)
df_segment.head()

(1013312, 5)


,group_tc,customer_id,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment
0,control,5c408e214a267149c7727449,RARE_NEED,UNKNOWN,DORMANT
1,control,5d49159dbde10e0d542eefe1,MONTHLY,UNKNOWN,DORMANT
2,control,618104ee106863646e2a1449,MONTHLY,UNKNOWN,SILVER
3,control,62513fe217073f7e00bd84ae,RARE_NEED,UNKNOWN,INACTIVE
4,control,62d845d06af03d2d8bc1862a,RARE_NEED,UNKNOWN,SILVER


## Adding Derived data

In [14]:
# Adding required fields
df_viewed_cux['group_tc_flag'] = 'viewed'
df_orders['time_window'] = np.where(df_orders['yyyymmdd'].isin(['20250405','20250406','20250407','20250408','20250409','20250410','20250411']), 
                                    'pre-period',
                                    np.where(df_orders['yyyymmdd'].isin(['20250412','20250413','20250414','20250415','20250416','20250417','20250418','20250419']),
                                             'exp-period',
                                             'post-period'))



In [15]:
display(df_orders.head(1))
display(df_segment.head(1))
display(df_viewed_cux.head(1))

,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window
0,test,20250413,5a967a8515e10f72cdf433a2,Bike Lite,67fbbb074c6f591c05e18197,dropped,NaN,1744550663081,1.744551e+12,NaN,NaN,NaN,NaN,1744550663081,1845,185423,4.63,1.072,exp-period


,group_tc,customer_id,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment
0,control,5c408e214a267149c7727449,RARE_NEED,UNKNOWN,DORMANT


,userid,group_tc_flag
0,63882d5a08087f9f39a201fe,viewed


In [16]:
df_orders.groupby(['time_window']).agg(min = ('yyyymmdd', 'min'), max =('yyyymmdd', 'max'), count = ('yyyymmdd', 'nunique')).reset_index()

,time_window,min,max,count
0,exp-period,20250412,20250419,8
1,post-period,20250420,20250426,7
2,pre-period,20250405,20250411,7


In [17]:
# Merge 

df_merge = pd.merge(df_orders, df_segment[['customer_id', 'taxi_regularity_segment','taxi_service_affinity', 'taxi_retention_segment']], how='left', on='customer_id')
df_merge = pd.merge(df_merge, df_viewed_cux, how='left', left_on='customer_id', right_on='userid')


df_merge['group_tc_flag'] = np.where(df_merge['group_tc'] == 'control', 'control', df_merge['group_tc_flag'].fillna(df_merge['group_tc']))

df_merge['group_tc_flag'] = np.where((df_merge['time_window'].isin(['post-period', 'pre-period'])) & (df_merge['group_tc_flag'] == 'viewed' ), 'test', df_merge['group_tc_flag'])

df_merge.head(1)

,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment,userid,group_tc_flag
0,test,20250413,5a967a8515e10f72cdf433a2,Bike Lite,67fbbb074c6f591c05e18197,dropped,NaN,1744550663081,1.744551e+12,NaN,NaN,NaN,NaN,1744550663081,1845,185423,4.63,1.072,exp-period,WEEKLY,ONLY_LINK,PRIME,5a967a8515e10f72cdf433a2,viewed


In [83]:
# Define bin edges and labels (in km)
bins = [0.0, 0.5, 1.0, 2.0, np.inf]
labels = ['a. 0-500M', 'b. 501M-1KM', 'c. 01-2KM', 'd. 2KM+']

# Apply binning
df_merge['pickup_distance_bucket'] = pd.cut(df_merge['accept_to_pickup_distance'], 
                                            bins=bins, 
                                            labels=labels,
                                            right=True,
                                            include_lowest=True)

df_merge['pickup_distance_bucket'] = df_merge['pickup_distance_bucket'].astype(object).fillna('NA')

In [84]:
# Define conditions and choices
conditions = [
    (df_merge['quarter_hour'] >= '0800') & (df_merge['quarter_hour'] < '1100'),
    (df_merge['quarter_hour'] >= '1100') & (df_merge['quarter_hour'] < '1700'),
    (df_merge['quarter_hour'] >= '1700') & (df_merge['quarter_hour'] < '2100'),
]
choices = ['1. morning', '2. afternoon', '3. evening']

# Apply bucketing with np.select
df_merge['time_temporal'] = np.select(conditions, choices, default='4. rest')

In [85]:
df_merge.groupby(['time_window', 'group_tc', 'group_tc_flag']).agg(yyyymmdd=('yyyymmdd', 'min'))

yyyymmdd
time_window group_tc group_tc_flag          
exp-period  control  control        20250412
            test     test           20250412
                     viewed         20250412
post-period control  control        20250420
            test     test           20250420
pre-period  control  control        20250405
            test     test           20250405

#### Filtering experimental period

In [86]:
df_merge_exp_period = df_merge[df_merge['time_window'] == 'exp-period']

# Sort and create row_number
df_merge_exp_period = df_merge_exp_period.sort_values(['customer_id', 'epoch'])
df_merge_exp_period['row_number'] = df_merge_exp_period.groupby('customer_id').cumcount() + 1

df_merge_exp_period['rr_rank'] = np.where(df_merge_exp_period['row_number'] == 1, '1st rr', np.where(df_merge_exp_period['row_number'] == 2, '2nd rr', '2plus rr'))

In [87]:
df_merge_exp_period.head(5)

,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment,userid,group_tc_flag,pickup_distance_bucket,time_temporal,row_number,rr_rank
3143745,test,20250416,5737c6baddbec2203f7331d9,Link,67ff3acee888171d90fa7b30,dropped,NaN,1744779982040,1.744780e+12,NaN,NaN,NaN,NaN,1744779982040,1030,103622,4.841154,0.530,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,b. 501M-1KM,1. morning,1,1st rr
3143746,test,20250419,5737c6baddbec2203f7331d9,Link,68033311f15bed13e145d6d2,dropped,NaN,1745040145019,1.745040e+12,NaN,NaN,NaN,NaN,1745040145019,1045,105225,3.040000,1.377,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,c. 01-2KM,1. morning,2,2nd rr
739258,control,20250419,5737c6e3ddbec2203f733341,Link,6803553d39dca47f7e594927,dropped,NaN,1745048893140,1.745049e+12,NaN,NaN,NaN,NaN,1745048893140,1315,131813,2.700000,0.226,exp-period,DAILY,LINK_AUTO,ELITE,NaN,control,a. 0-500M,2. afternoon,1,1st rr
739259,control,20250419,5737c6e3ddbec2203f733341,Link,68035ee38bd4380737cf7156,OCARA,1.745052e+12,1745051363720,1.745052e+12,0.905014,NaN,467.0,0.0,1745051363720,1345,135923,4.550000,0.686,exp-period,DAILY,LINK_AUTO,ELITE,NaN,control,b. 501M-1KM,2. afternoon,2,2nd rr
739261,control,20250419,5737c6e3ddbec2203f733341,Link,680361bf2859a3306f4ed6a8,dropped,NaN,1745052095543,1.745052e+12,NaN,NaN,NaN,NaN,1745052095543,1400,141135,4.090000,0.031,exp-period,DAILY,LINK_AUTO,ELITE,NaN,control,a. 0-500M,2. afternoon,3,2plus rr


### Checking distributions

In [23]:
display(df_merge_exp_period.groupby(['group_tc']).agg(customers = ('customer_id', 'nunique')).reset_index())

,group_tc,customers
0,control,113463
1,test,196046


In [24]:
df_merge_exp_period.groupby(['group_tc', 'group_tc_flag']).agg(customers = ('customer_id', 'nunique')).reset_index()

,group_tc,group_tc_flag,customers
0,control,control,113463
1,test,test,96787
2,test,viewed,99259


In [25]:
df_temp = df_merge_exp_period.groupby(['group_tc','taxi_regularity_segment']).agg(customers = ('customer_id', 'nunique')).reset_index()

pivot = df_temp.pivot(index="taxi_regularity_segment", columns="group_tc", values="customers")

display(pivot)

pivot_percent = pivot.div(pivot.sum(), axis=1) * 100
pivot_percent = pivot_percent.round(2)
display(pivot_percent)

group_tc,control,test
taxi_regularity_segment,,
BI_WEEKLY,27072,46790
DAILY,16824,29409
MONTHLY,14614,25702
RARE_NEED,10037,17476
UNKNOWN,10085,15555
WEEKLY,34831,61110


group_tc,control,test
taxi_regularity_segment,,
BI_WEEKLY,23.86,23.87
DAILY,14.83,15.00
MONTHLY,12.88,13.11
RARE_NEED,8.85,8.91
UNKNOWN,8.89,7.93
WEEKLY,30.70,31.17


In [26]:
df_temp = df_merge_exp_period.groupby(['group_tc', 'group_tc_flag','taxi_regularity_segment']).agg(customers = ('customer_id', 'nunique')).reset_index()

pivot = df_temp.pivot(index=["taxi_regularity_segment"], columns="group_tc_flag", values="customers")

display(pivot)

pivot_percent = pivot.div(pivot.sum(), axis=1) * 100
pivot_percent = pivot_percent.round(2)

display(pivot_percent)

group_tc_flag,control,test,viewed
taxi_regularity_segment,,,
BI_WEEKLY,27072,25528,21262
DAILY,16824,8353,21056
MONTHLY,14614,15223,10479
RARE_NEED,10037,10566,6910
UNKNOWN,10085,9108,6447
WEEKLY,34831,28006,33104


group_tc_flag,control,test,viewed
taxi_regularity_segment,,,
BI_WEEKLY,23.86,26.38,21.42
DAILY,14.83,8.63,21.21
MONTHLY,12.88,15.73,10.56
RARE_NEED,8.85,10.92,6.96
UNKNOWN,8.89,9.41,6.50
WEEKLY,30.70,28.94,33.35


In [27]:
df_temp = df_merge_exp_period.groupby(['group_tc','taxi_retention_segment']).agg(customers = ('customer_id', 'nunique')).reset_index()

pivot = df_temp.pivot(index=["taxi_retention_segment"], columns="group_tc", values="customers")

display(pivot)

pivot_percent = pivot.div(pivot.sum(), axis=1) * 100
pivot_percent = pivot_percent.round(2)

display(pivot_percent)

group_tc,control,test
taxi_retention_segment,,
DORMANT,22803,39707
ELITE,22591,39745
GOLD,21747,37469
HH,5112,7390
INACTIVE,2493,4914
PLATINUM,25680,44655
PRIME,417,728
SILVER,11925,20417
UNKNOWN,695,1017


group_tc,control,test
taxi_retention_segment,,
DORMANT,20.10,20.25
ELITE,19.91,20.27
GOLD,19.17,19.11
HH,4.51,3.77
INACTIVE,2.20,2.51
PLATINUM,22.63,22.78
PRIME,0.37,0.37
SILVER,10.51,10.41
UNKNOWN,0.61,0.52


In [28]:
df_temp = df_merge_exp_period.groupby(['group_tc', 'group_tc_flag','taxi_retention_segment']).agg(customers = ('customer_id', 'nunique')).reset_index()

pivot = df_temp.pivot(index=["taxi_retention_segment"], columns="group_tc_flag", values="customers")

display(pivot)

pivot_percent = pivot.div(pivot.sum(), axis=1) * 100
pivot_percent = pivot_percent.round(2)

display(pivot_percent)

group_tc_flag,control,test,viewed
taxi_retention_segment,,,
DORMANT,22803,23989,15718
ELITE,22591,12224,27521
GOLD,21747,20047,17422
HH,5112,4071,3319
INACTIVE,2493,3156,1758
PLATINUM,25680,20441,24214
PRIME,417,341,387
SILVER,11925,11899,8518
UNKNOWN,695,616,401


group_tc_flag,control,test,viewed
taxi_retention_segment,,,
DORMANT,20.10,24.79,15.84
ELITE,19.91,12.63,27.73
GOLD,19.17,20.71,17.55
HH,4.51,4.21,3.34
INACTIVE,2.20,3.26,1.77
PLATINUM,22.63,21.12,24.40
PRIME,0.37,0.35,0.39
SILVER,10.51,12.29,8.58
UNKNOWN,0.61,0.64,0.40


### Resample 

In [52]:
def resampling(df):

        # 1. Get only unique control customers
        control_customers = df[df['group_tc_flag'] == 'control'][['customer_id', 'taxi_regularity_segment']].drop_duplicates()

        # 2. Prepare your viewed-based target distribution
        distribution = {
            'taxi_regularity_segment': ['BI_WEEKLY', 'DAILY', 'MONTHLY', 'RARE_NEED', 'UNKNOWN', 'WEEKLY'],
            'control': [23.86, 14.83, 12.88, 8.85, 8.89, 30.70],
            'viewed': [21.42, 21.21, 10.56, 6.96, 6.50, 33.35]
        }

        dist_df = pd.DataFrame(distribution)
        dist_df['viewed_norm'] = dist_df['viewed'] / dist_df['viewed'].sum()

        # 3. Calculate how many customers per segment you need
        target_total = len(control_customers)
        dist_df['target_count'] = (dist_df['viewed_norm'] * target_total).round().astype(int)

        # 4. Resample customers
        resampled_customers = pd.DataFrame()

        for segment, count in zip(dist_df['taxi_regularity_segment'], dist_df['target_count']):
            segment_customers = control_customers[control_customers['taxi_regularity_segment'] == segment]
            
            if len(segment_customers) >= count:
                sampled = segment_customers.sample(n=count, random_state=42)
            else:
                sampled = segment_customers.sample(n=count, replace=True, random_state=42)
                
            resampled_customers = pd.concat([resampled_customers, sampled])
        
        # 5. Now, if you want back **order-level** data for these customers:
        final_resampled_df = df[df['customer_id'].isin(resampled_customers['customer_id'])]



        # 1. How many unique control customers per segment
        available_customers = control_customers['taxi_regularity_segment'].value_counts()
        
        # 2. Target customers needed per segment
        target_customers = dist_df.set_index('taxi_regularity_segment')['target_count']
        
        # 3. Compare side-by-side
        comparison = pd.DataFrame({
            'available': available_customers,
            'target': target_customers
        }).fillna(0).astype(int)
        
        print(comparison)

        
        return final_resampled_df

In [53]:
final_resampled_df = resampling(df_merge_exp_period)
final_resampled_df

                         available  target
taxi_regularity_segment                   
BI_WEEKLY                    27072   24304
DAILY                        16824   24066
MONTHLY                      14614   11982
RARE_NEED                    10037    7897
UNKNOWN                      10085    7375
WEEKLY                       34831   37840


,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment,userid,group_tc_flag,pickup_distance_bucket,time_temporal,row_number,rr_rank
2123894,control,20250415,5737c732ddbec2203f7335f0,Link,67fde53cc819dc3c5625e854,dropped,NaN,1744692540036,1.744693e+12,NaN,NaN,NaN,NaN,1744692540036,1015,101900,10.530,2.148,exp-period,WEEKLY,ONLY_LINK,PLATINUM,NaN,control,2KM+,1. morning,1,1st rr
2123884,control,20250416,5737c732ddbec2203f7335f0,Link,67ff18624cf38a538d5892c5,COBRA,1.744771e+12,1744771170636,NaN,NaN,102.0,NaN,NaN,1744771170636,0800,080930,6.630,NaN,exp-period,WEEKLY,ONLY_LINK,PLATINUM,NaN,control,NA,1. morning,2,2nd rr
2123887,control,20250417,5737c732ddbec2203f7335f0,Link,6801365058398c0b370721cc,dropped,NaN,1744909904549,1.744910e+12,NaN,NaN,NaN,NaN,1744909904549,2230,224144,3.864,2.252,exp-period,WEEKLY,ONLY_LINK,PLATINUM,NaN,control,2KM+,4. rest,3,2plus rr
3192937,control,20250416,573f28f39b0ffc283676f0b5,Auto,67ff35a210d03d09b7cd0333,dropped,NaN,1744778658495,1.744779e+12,NaN,NaN,NaN,NaN,1744778658495,1000,101418,2.790,0.517,exp-period,RARE_NEED,UNKNOWN,DORMANT,NaN,control,501M-1KM,1. morning,1,1st rr
2112806,control,20250414,573f28f39b0ffc283676f0c5,CabEconomy,67fc76c8ec8e821585d63218,dropped,NaN,1744598728136,1.744599e+12,NaN,NaN,NaN,NaN,1744598728136,0815,081528,29.900,2.369,exp-period,BI_WEEKLY,ONLY_CAB,GOLD,NaN,control,2KM+,1. morning,1,1st rr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17004,control,20250419,67f69412d25c76285ed03200,Link,68030ed29cfbc45127ba2c12,dropped,NaN,1745030866439,1.745031e+12,NaN,NaN,NaN,NaN,1745030866439,0815,081746,3.990,0.452,exp-period,UNKNOWN,UNKNOWN,HH,NaN,control,0-500M,1. morning,8,2plus rr
16983,control,20250419,67f69412d25c76285ed03200,Auto,6803b55d07597a4de3b306bf,dropped,NaN,1745073501047,1.745074e+12,NaN,NaN,NaN,NaN,1745073501047,2000,200821,4.440,0.983,exp-period,UNKNOWN,UNKNOWN,HH,NaN,control,501M-1KM,3. evening,9,2plus rr
2761043,control,20250412,67f699b129fd3eec011ac2b1,CabEconomy,67fa915c78a15f490d2ea967,dropped,NaN,1744474460580,1.744474e+12,NaN,NaN,NaN,NaN,1744474460580,2130,214420,18.030,1.704,exp-period,UNKNOWN,UNKNOWN,HH,NaN,control,1.01-2KM,4. rest,1,1st rr
176628,control,20250412,67f6af6dedbb784d97740f17,Auto,67f9de42d9a4a17e84349ca3,dropped,NaN,1744428610041,1.744429e+12,NaN,NaN,NaN,NaN,1744428610041,0900,090010,3.560,2.590,exp-period,RARE_NEED,ONLY_AUTO,GOLD,NaN,control,2KM+,1. morning,1,1st rr


In [54]:
df_temp = final_resampled_df.groupby(['group_tc', 'group_tc_flag','taxi_regularity_segment']).agg(customers = ('customer_id', 'nunique')).reset_index()

pivot = df_temp.pivot(index=["taxi_regularity_segment"], columns="group_tc_flag", values="customers")

display(pivot)

pivot_percent = pivot.div(pivot.sum(), axis=1) * 100
pivot_percent = pivot_percent.round(2)
display(pivot_percent)

group_tc_flag,control,test,viewed
taxi_regularity_segment,,,
BI_WEEKLY,24304,952,490
DAILY,12778,271,428
MONTHLY,11982,513,241
RARE_NEED,7897,331,105
UNKNOWN,7375,240,103
WEEKLY,23041,797,605


group_tc_flag,control,test,viewed
taxi_regularity_segment,,,
BI_WEEKLY,27.82,30.67,24.85
DAILY,14.62,8.73,21.70
MONTHLY,13.71,16.53,12.22
RARE_NEED,9.04,10.66,5.32
UNKNOWN,8.44,7.73,5.22
WEEKLY,26.37,25.68,30.68


## Analysis 

### Analysis

In [29]:
df_merge_exp_period.head(1)

,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment,userid,group_tc_flag,pickup_distance_bucket,time_temporal,row_number,rr_rank
3143745,test,20250416,5737c6baddbec2203f7331d9,Link,67ff3acee888171d90fa7b30,dropped,NaN,1744779982040,1.744780e+12,NaN,NaN,NaN,NaN,1744779982040,1030,103622,4.841154,0.53,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,501M-1KM,1. morning,1,1st rr


In [30]:
df_merge_exp_period.modified_order_status.unique()

array(['dropped', 'OCARA', 'COBRA', 'expired', 'COBRM', 'aborted',
       'requested', 'new'], dtype=object)

In [31]:
def get_funnel(df, grouper):
    df_analysis =  df\
    .groupby(grouper)\
    .agg(gross_customer = ('customer_id', 'nunique'),
         gross_orders = ('order_id', 'nunique'),
         net_orders = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'dropped'].nunique()),
         cobra = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'COBRA'].nunique()),
         ocara = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'OCARA'].nunique()),
         cobrm = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'COBRM'].nunique()),
         expired = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'expired'].nunique()),
         aborted = ('order_id', lambda x: x[df.loc[x.index, 'modified_order_status'] == 'aborted'].nunique()),
         
         cobra_ttc_mean = ('cobra_ttc', 'mean'),
         cobra_ttc_p25=('cobra_ttc', lambda x: x.quantile(0.25)),
         cobra_ttc_p50 = ('cobra_ttc', 'median'),
         cobra_ttc_p75=('cobra_ttc', lambda x: x.quantile(0.75)),
         cobra_ttc_p90=('cobra_ttc', lambda x: x.quantile(0.90)),
         cobra_ttc_p95=('cobra_ttc', lambda x: x.quantile(0.95)),
    
         ocara_ttc_mean = ('ocara_ttc', 'mean'),
         ocara_ttc_p25=('ocara_ttc', lambda x: x.quantile(0.25)),
         ocara_ttc_p50 = ('ocara_ttc', 'median'),
         ocara_ttc_p75=('ocara_ttc', lambda x: x.quantile(0.75)),
         ocara_ttc_p90=('ocara_ttc', lambda x: x.quantile(0.90)),
         ocara_ttc_p95=('ocara_ttc', lambda x: x.quantile(0.95)),
         
        )\
    .reset_index()

    df_analysis['cobra_ttc_mean'] = (df_analysis['cobra_ttc_mean']).round(1)
    df_analysis['ocara_ttc_mean'] = (df_analysis['ocara_ttc_mean']).round(1)
    
    df_analysis['g2n %'] = (df_analysis['net_orders']*100.00/df_analysis['gross_orders']).round(2)
    
    df_analysis['cobra %'] = (df_analysis['cobra']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['ocara %'] = (df_analysis['ocara']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['cobrn %'] = (df_analysis['cobrm']*100.00/df_analysis['gross_orders']).round(2)
    df_analysis['expired %'] = ((df_analysis['expired'] + df_analysis['aborted'] )*100.00/df_analysis['gross_orders']).round(2)


    columns = grouper + ['gross_customer', 'gross_orders', 'net_orders', 'g2n %', 'cobra %', 'ocara %', 'cobrn %', 'expired %',
               'cobra_ttc_mean', 'cobra_ttc_p25', 'cobra_ttc_p50', 'cobra_ttc_p75', 'cobra_ttc_p90', 'cobra_ttc_p95', 
               'ocara_ttc_mean', 'ocara_ttc_p25', 'ocara_ttc_p50', 'ocara_ttc_p75', 'ocara_ttc_p90', 'ocara_ttc_p95', 
               'cobrm', 'cobra', 'ocara', 'expired', 'aborted']

    return df_analysis[columns]

In [32]:
get_funnel(df_merge_exp_period, ['group_tc'])

,group_tc,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,control,113463,511853,254284,49.68,24.29,14.83,0.61,10.60,127.8,55.0,105.0,181.0,262.0,302.0,215.4,43.0,119.0,284.0,525.0,709.0,3098,124305,75912,53943,302
1,test,196046,875236,436655,49.89,24.10,14.86,0.59,10.56,128.6,55.0,106.0,183.0,263.0,303.0,212.1,42.0,119.0,282.0,517.0,700.0,5168,210890,130086,91942,476


In [33]:
get_funnel(df_merge_exp_period, ['group_tc','rr_rank'])

,group_tc,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,control,1st rr,111551,111551,64930,58.21,17.47,15.51,0.44,8.38,133.3,59.0,113.0,188.0,267.0,307.0,240.0,48.0,135.0,323.0,581.0,764.25,486,19487,17296,9264,85
1,control,2nd rr,79340,79340,41897,52.81,21.77,15.18,0.54,9.71,128.3,55.0,106.0,181.0,262.0,304.0,230.1,46.0,127.0,298.0,546.0,742.00,425,17275,12041,7645,56
2,control,2plus rr,59799,325660,150198,46.12,27.13,14.53,0.68,11.54,126.6,54.0,103.0,180.0,261.0,302.0,203.6,40.0,112.0,267.0,497.0,676.00,2206,88349,47330,37410,162
3,test,1st rr,191348,191348,111735,58.39,17.34,15.46,0.46,8.34,134.4,60.0,114.0,190.0,268.0,307.0,239.5,49.0,137.0,322.0,577.0,763.50,877,33182,29591,15807,151
4,test,2nd rr,134543,134543,71164,52.89,21.70,15.08,0.54,9.78,130.0,56.0,108.0,184.0,265.0,304.0,223.3,44.0,124.0,296.0,547.0,729.00,728,29199,20292,13055,101
5,test,2plus rr,103751,558945,259232,46.38,26.87,14.62,0.65,11.48,127.2,54.0,104.0,181.0,262.0,302.0,201.0,39.0,112.0,266.0,489.3,668.15,3622,150209,81718,63920,234


In [34]:
get_funnel(df_merge_exp_period, ['group_tc_flag','rr_rank'])

,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,control,1st rr,111551,111551,64930,58.21,17.47,15.51,0.44,8.38,133.3,59.0,113.0,188.0,267.0,307.0,240.0,48.0,135.0,323.0,581.0,764.25,486,19487,17296,9264,85
1,control,2nd rr,79340,79340,41897,52.81,21.77,15.18,0.54,9.71,128.3,55.0,106.0,181.0,262.0,304.0,230.1,46.0,127.0,298.0,546.0,742.00,425,17275,12041,7645,56
2,control,2plus rr,59799,325660,150198,46.12,27.13,14.53,0.68,11.54,126.6,54.0,103.0,180.0,261.0,302.0,203.6,40.0,112.0,267.0,497.0,676.00,2206,88349,47330,37410,162
3,test,1st rr,93934,93934,54426,57.94,18.09,14.85,0.56,8.55,132.8,59.0,113.0,187.0,264.0,305.0,251.7,52.0,147.0,345.0,607.0,793.00,525,16994,13949,7930,106
4,test,2nd rr,54979,54979,28753,52.30,22.78,14.17,0.65,10.10,127.4,56.0,106.0,181.0,258.6,299.0,243.7,48.0,132.0,314.0,576.0,764.00,356,12525,7789,5492,62
5,test,2plus rr,37095,131903,59412,45.04,28.58,13.75,0.77,11.85,125.3,54.0,103.0,178.0,257.0,297.0,222.7,43.0,121.0,292.0,535.0,730.70,1014,37698,18140,15544,92
6,viewed,1st rr,97414,97414,57309,58.83,16.62,16.06,0.36,8.13,136.0,60.0,116.0,192.0,271.0,310.0,228.6,46.0,130.0,306.0,551.0,739.95,352,16188,15642,7877,45
7,viewed,2nd rr,79564,79564,42411,53.30,20.96,15.71,0.47,9.55,131.9,56.0,110.0,187.0,270.0,309.0,210.6,42.0,119.0,284.0,524.0,701.00,372,16674,12503,7563,39
8,viewed,2plus rr,66656,427042,199820,46.79,26.35,14.89,0.61,11.36,127.8,54.0,104.0,182.0,263.0,304.0,194.8,38.0,109.0,259.0,476.0,651.00,2608,112511,63578,48376,142


In [35]:
get_funnel(df_merge_exp_period[df_merge_exp_period['rr_rank'] == '1st rr'], ['group_tc_flag','rr_rank'])

,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,control,1st rr,111551,111551,64930,58.21,17.47,15.51,0.44,8.38,133.3,59.0,113.0,188.0,267.0,307.0,240.0,48.0,135.0,323.0,581.0,764.25,486,19487,17296,9264,85
1,test,1st rr,93934,93934,54426,57.94,18.09,14.85,0.56,8.55,132.8,59.0,113.0,187.0,264.0,305.0,251.7,52.0,147.0,345.0,607.0,793.00,525,16994,13949,7930,106
2,viewed,1st rr,97414,97414,57309,58.83,16.62,16.06,0.36,8.13,136.0,60.0,116.0,192.0,271.0,310.0,228.6,46.0,130.0,306.0,551.0,739.95,352,16188,15642,7877,45


In [36]:
get_funnel(df_merge_exp_period[df_merge_exp_period['rr_rank'] == '1st rr'], ['yyyymmdd', 'group_tc_flag','rr_rank'])

,yyyymmdd,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,20250412,control,1st rr,29612,29612,17932,60.56,15.87,16.57,0.32,6.68,129.4,57.00,109.0,181.00,263.0,301.00,231.2,44.0,125.0,308.25,566.0,754.00,94,4700,4908,1961,17
1,20250412,test,1st rr,16976,16976,10066,59.30,17.49,15.93,0.35,6.93,125.7,56.00,107.0,176.00,254.0,288.60,247.0,49.0,147.0,339.25,606.7,778.00,60,2969,2704,1153,24
2,20250412,viewed,1st rr,33800,33800,20817,61.59,15.19,16.70,0.28,6.25,127.5,54.00,107.0,182.00,259.0,298.00,221.0,44.0,126.0,291.00,529.7,721.85,93,5134,5644,2100,12
3,20250413,control,1st rr,16579,16579,10123,61.06,15.21,16.58,0.36,6.80,128.8,56.00,107.0,183.00,263.0,300.00,238.9,48.0,133.0,314.50,574.6,736.30,60,2521,2748,1112,15
4,20250413,test,1st rr,10847,10847,6447,59.44,16.96,15.68,0.57,7.33,125.4,54.00,107.0,176.00,251.1,295.00,261.7,52.0,157.0,362.00,605.0,812.00,62,1840,1701,772,23
5,20250413,viewed,1st rr,17464,17464,10705,61.30,15.15,16.54,0.33,6.68,131.4,56.25,111.0,185.00,264.0,306.00,218.5,42.0,119.0,295.00,543.3,714.65,57,2646,2888,1158,9
6,20250414,control,1st rr,15696,15696,9917,63.18,14.78,14.83,0.29,6.92,128.5,55.00,106.0,181.00,261.0,305.00,240.1,49.0,139.0,327.00,571.4,772.00,46,2320,2327,1072,14
7,20250414,test,1st rr,12365,12365,7620,61.63,16.05,14.65,0.41,7.26,131.8,57.00,111.0,187.00,260.0,303.85,239.0,47.0,137.0,322.50,594.8,788.45,51,1984,1812,888,10
8,20250414,viewed,1st rr,14342,14342,9302,64.86,13.16,15.31,0.28,6.39,134.4,62.00,116.0,188.25,268.0,304.30,234.2,49.0,135.0,319.00,544.0,718.75,40,1888,2196,906,10
9,20250415,control,1st rr,12911,12911,7484,57.97,17.54,15.78,0.26,8.45,133.7,59.00,116.0,191.00,268.0,303.00,237.7,48.0,136.0,324.00,583.3,760.15,33,2265,2038,1081,10


In [37]:
get_funnel(df_merge_exp_period[df_merge_exp_period['rr_rank'] == '1st rr'], ['time_window', 'taxi_regularity_segment', 'group_tc_flag','rr_rank'])

,time_window,taxi_regularity_segment,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,exp-period,BI_WEEKLY,control,1st rr,26584,26584,15395,57.91,18.23,15.17,0.48,8.20,133.4,61.0,112.0,186.00,265.0,310.75,240.1,50.00,138.0,338.25,588.0,762.45,128,4846,4032,2153,28
1,exp-period,BI_WEEKLY,test,1st rr,24789,24789,14499,58.49,18.33,14.40,0.55,8.22,132.1,60.0,111.0,184.00,262.0,304.80,255.3,57.00,147.0,344.00,624.0,809.00,137,4545,3570,2005,33
2,exp-period,BI_WEEKLY,viewed,1st rr,20871,20871,12023,57.61,17.45,16.06,0.41,8.48,137.7,63.0,120.0,192.00,271.0,313.00,234.6,52.00,135.0,313.00,564.0,744.50,85,3643,3351,1759,10
3,exp-period,DAILY,control,1st rr,16577,16577,10248,61.82,15.11,16.13,0.24,6.70,133.8,54.0,113.0,188.00,272.0,315.80,192.1,34.00,96.0,248.00,464.4,658.70,40,2505,2674,1106,4
4,exp-period,DAILY,test,1st rr,8092,8092,4740,58.58,17.62,15.62,0.54,7.64,135.2,56.0,115.0,195.75,272.0,307.75,201.6,34.00,104.0,276.25,489.7,671.45,44,1426,1264,607,11
5,exp-period,DAILY,viewed,1st rr,20638,20638,12894,62.48,14.35,16.44,0.25,6.49,127.9,51.0,108.0,186.00,264.0,299.00,190.5,34.00,104.0,263.00,463.9,633.80,51,2961,3392,1334,6
6,exp-period,MONTHLY,control,1st rr,14338,14338,8374,58.40,17.54,14.76,0.47,8.82,136.1,62.0,115.0,194.00,271.0,306.00,260.5,59.00,161.0,348.00,623.4,789.40,68,2515,2117,1251,13
7,exp-period,MONTHLY,test,1st rr,14771,14771,8671,58.70,17.65,14.51,0.62,8.52,135.5,61.0,117.0,188.00,267.0,306.70,273.9,56.00,163.0,372.00,632.0,804.40,92,2607,2143,1243,15
8,exp-period,MONTHLY,viewed,1st rr,10288,10288,5977,58.10,17.16,15.13,0.46,9.16,142.0,67.0,126.0,197.00,274.2,309.60,295.0,60.00,160.0,359.00,642.0,852.20,47,1765,1557,940,2
9,exp-period,RARE_NEED,control,1st rr,9877,9877,5686,57.57,17.98,14.90,0.56,8.98,136.8,64.0,117.5,193.00,269.0,302.00,288.5,53.75,149.0,386.25,672.9,862.60,55,1776,1472,872,15


In [38]:
get_funnel(df_merge[df_merge['time_window'].isin(['pre-period', 'post-period'])], ['time_window', 'taxi_regularity_segment', 'group_tc_flag'])

,time_window,taxi_regularity_segment,group_tc_flag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,post-period,BI_WEEKLY,control,22121,64348,36104,56.11,20.48,15.25,0.99,7.18,121.0,53.0,100.0,170.00,247.8,290.0,227.2,46.00,129.0,308.00,555.0,737.00,636,13176,9812,4576,44
1,post-period,BI_WEEKLY,test,38598,113203,63268,55.89,20.83,15.08,0.88,7.32,119.1,51.0,97.0,168.00,246.0,289.0,218.6,44.00,125.0,297.00,534.0,717.00,993,23582,17069,8211,78
2,post-period,DAILY,control,15856,132602,75833,57.19,19.05,15.76,0.56,7.45,117.9,47.0,97.0,170.00,247.0,289.0,177.1,33.00,94.0,237.00,438.0,588.00,740,25260,20892,9829,48
3,post-period,DAILY,test,27807,232213,132066,56.87,19.37,15.90,0.55,7.30,118.4,48.0,97.0,171.00,250.0,289.0,177.1,33.00,95.0,236.00,434.0,598.70,1279,44989,36931,16848,97
4,post-period,MONTHLY,control,11968,30210,17187,56.89,20.27,14.57,0.96,7.30,121.6,54.0,101.0,170.00,247.0,289.0,241.4,47.00,137.0,326.50,591.4,771.70,289,6125,4403,2180,26
5,post-period,MONTHLY,test,20787,51788,29572,57.10,19.85,14.78,0.95,7.33,122.0,54.0,101.0,170.00,248.0,292.0,241.2,53.00,137.0,328.00,580.9,748.00,490,10278,7654,3751,43
6,post-period,RARE_NEED,control,8218,21548,12094,56.13,20.98,14.45,0.96,7.49,119.1,51.0,97.0,169.00,248.0,285.0,243.5,51.00,140.0,326.25,582.0,779.50,207,4520,3113,1597,17
7,post-period,RARE_NEED,test,14745,38488,21912,56.93,20.34,14.28,0.91,7.54,120.9,52.0,100.0,172.00,247.0,292.0,233.8,50.00,137.0,321.00,555.6,743.80,350,7828,5496,2873,29
8,post-period,UNKNOWN,control,8055,19518,10369,53.13,19.59,16.71,1.16,9.41,122.9,53.0,100.0,172.00,251.0,293.0,276.9,59.00,163.0,389.00,650.0,861.05,227,3824,3261,1813,24
9,post-period,UNKNOWN,test,12524,28732,15229,53.00,19.47,16.52,1.28,9.72,123.0,52.0,100.0,171.00,256.0,296.0,272.6,56.00,155.0,374.00,655.0,845.50,368,5595,4747,2751,42


### Regularity

In [39]:
df1 = get_funnel(df_merge_exp_period[df_merge_exp_period['rr_rank'] == '1st rr'], ['time_window', 'taxi_regularity_segment', 'group_tc_flag','rr_rank'])
df2 = get_funnel(df_merge[df_merge['time_window'].isin(['pre-period', 'post-period'])], ['time_window', 'taxi_regularity_segment', 'group_tc_flag'])

In [40]:
df1[['time_window',	'taxi_regularity_segment',	'group_tc_flag',	'rr_rank', 
        'g2n %',	'cobra %',	'ocara %',
        'cobra_ttc_mean', 'cobra_ttc_p50', 'cobra_ttc_p75', 'ocara_ttc_mean', 'ocara_ttc_p50', 'ocara_ttc_p75']]

,time_window,taxi_regularity_segment,group_tc_flag,rr_rank,g2n %,cobra %,ocara %,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75
0,exp-period,BI_WEEKLY,control,1st rr,57.91,18.23,15.17,133.4,112.0,186.00,240.1,138.0,338.25
1,exp-period,BI_WEEKLY,test,1st rr,58.49,18.33,14.40,132.1,111.0,184.00,255.3,147.0,344.00
2,exp-period,BI_WEEKLY,viewed,1st rr,57.61,17.45,16.06,137.7,120.0,192.00,234.6,135.0,313.00
3,exp-period,DAILY,control,1st rr,61.82,15.11,16.13,133.8,113.0,188.00,192.1,96.0,248.00
4,exp-period,DAILY,test,1st rr,58.58,17.62,15.62,135.2,115.0,195.75,201.6,104.0,276.25
5,exp-period,DAILY,viewed,1st rr,62.48,14.35,16.44,127.9,108.0,186.00,190.5,104.0,263.00
6,exp-period,MONTHLY,control,1st rr,58.40,17.54,14.76,136.1,115.0,194.00,260.5,161.0,348.00
7,exp-period,MONTHLY,test,1st rr,58.70,17.65,14.51,135.5,117.0,188.00,273.9,163.0,372.00
8,exp-period,MONTHLY,viewed,1st rr,58.10,17.16,15.13,142.0,126.0,197.00,295.0,160.0,359.00
9,exp-period,RARE_NEED,control,1st rr,57.57,17.98,14.90,136.8,117.5,193.00,288.5,149.0,386.25


In [41]:
df2[['time_window',	'taxi_regularity_segment',	'group_tc_flag', 
        'g2n %',	'cobra %',	'ocara %',
        'cobra_ttc_mean', 'cobra_ttc_p50', 'cobra_ttc_p75', 'ocara_ttc_mean', 'ocara_ttc_p50', 'ocara_ttc_p75']]

,time_window,taxi_regularity_segment,group_tc_flag,g2n %,cobra %,ocara %,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75
0,post-period,BI_WEEKLY,control,56.11,20.48,15.25,121.0,100.0,170.00,227.2,129.0,308.00
1,post-period,BI_WEEKLY,test,55.89,20.83,15.08,119.1,97.0,168.00,218.6,125.0,297.00
2,post-period,DAILY,control,57.19,19.05,15.76,117.9,97.0,170.00,177.1,94.0,237.00
3,post-period,DAILY,test,56.87,19.37,15.90,118.4,97.0,171.00,177.1,95.0,236.00
4,post-period,MONTHLY,control,56.89,20.27,14.57,121.6,101.0,170.00,241.4,137.0,326.50
5,post-period,MONTHLY,test,57.10,19.85,14.78,122.0,101.0,170.00,241.2,137.0,328.00
6,post-period,RARE_NEED,control,56.13,20.98,14.45,119.1,97.0,169.00,243.5,140.0,326.25
7,post-period,RARE_NEED,test,56.93,20.34,14.28,120.9,100.0,172.00,233.8,137.0,321.00
8,post-period,UNKNOWN,control,53.13,19.59,16.71,122.9,100.0,172.00,276.9,163.0,389.00
9,post-period,UNKNOWN,test,53.00,19.47,16.52,123.0,100.0,171.00,272.6,155.0,374.00


### Retention

In [42]:
get_funnel(df_merge_exp_period[df_merge_exp_period['rr_rank'] == '1st rr'], ['time_window', 'taxi_retention_segment', 'group_tc_flag','rr_rank'])

,time_window,taxi_retention_segment,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,exp-period,DORMANT,control,1st rr,22422,22422,12559,56.01,18.28,15.36,0.62,9.73,133.1,61.00,113.0,188.00,261.0,304.00,277.6,55.00,155.0,363.25,652.7,846.85,138,4098,3444,2159,23
1,exp-period,DORMANT,test,1st rr,23247,23247,13112,56.40,18.29,15.21,0.65,9.43,135.3,61.00,116.0,189.00,267.0,306.00,272.9,61.00,172.0,374.50,632.6,829.00,152,4252,3535,2168,25
2,exp-period,DORMANT,viewed,1st rr,15476,15476,8592,55.52,18.01,16.00,0.51,9.95,142.7,66.00,125.0,199.00,275.0,317.00,255.7,57.75,156.0,340.25,612.5,812.75,79,2788,2476,1530,10
3,exp-period,ELITE,control,1st rr,22253,22253,14065,63.20,15.07,15.38,0.33,6.01,131.5,55.00,111.0,184.75,267.7,310.70,202.0,38.00,110.0,266.00,481.0,677.90,74,3354,3423,1329,8
4,exp-period,ELITE,test,1st rr,11843,11843,7313,61.75,16.77,14.61,0.52,6.35,130.8,53.00,112.0,189.00,261.5,298.00,217.7,39.00,121.0,297.00,537.1,705.30,62,1986,1730,737,15
5,exp-period,ELITE,viewed,1st rr,26982,26982,17197,63.74,14.14,15.95,0.29,5.89,128.1,53.00,108.0,185.00,262.0,301.30,192.5,37.00,109.0,264.00,466.8,625.80,77,3815,4303,1582,8
6,exp-period,GOLD,control,1st rr,21333,21333,12334,57.82,18.30,15.12,0.44,8.33,133.5,58.00,113.0,188.00,268.0,306.00,231.2,50.00,138.0,325.00,557.0,721.60,94,3903,3225,1751,25
7,exp-period,GOLD,test,1st rr,19456,19456,11319,58.18,18.24,14.57,0.57,8.45,131.8,60.00,112.0,185.00,264.0,302.00,252.5,53.00,142.0,347.75,609.0,787.70,110,3549,2834,1616,28
8,exp-period,GOLD,viewed,1st rr,17107,17107,9768,57.10,17.52,16.05,0.38,8.95,136.3,62.00,115.0,191.00,272.0,313.00,232.3,49.00,139.0,318.75,563.5,739.00,65,2997,2746,1524,7
9,exp-period,HH,control,1st rr,5053,5053,2871,56.82,15.83,16.01,0.51,10.83,140.0,65.00,119.5,190.25,275.2,319.30,329.1,73.00,205.0,445.00,701.0,907.40,26,800,809,536,11


In [43]:
df3 = get_funnel(df_merge_exp_period[df_merge_exp_period['rr_rank'] == '1st rr'], ['time_window', 'taxi_retention_segment', 'group_tc_flag','rr_rank'])
df4 = get_funnel(df_merge[df_merge['time_window'].isin(['pre-period'])], ['time_window', 'taxi_retention_segment', 'group_tc_flag'])

In [44]:
df3[~df3['group_tc_flag'].isin(['test'])][['time_window',	'taxi_retention_segment',	'group_tc_flag',	'rr_rank', 
        'g2n %',	'cobra %',	'ocara %',
        'cobra_ttc_mean', 'cobra_ttc_p50', 'cobra_ttc_p75', 'ocara_ttc_mean', 'ocara_ttc_p50', 'ocara_ttc_p75']]

,time_window,taxi_retention_segment,group_tc_flag,rr_rank,g2n %,cobra %,ocara %,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75
0,exp-period,DORMANT,control,1st rr,56.01,18.28,15.36,133.1,113.0,188.00,277.6,155.0,363.25
2,exp-period,DORMANT,viewed,1st rr,55.52,18.01,16.00,142.7,125.0,199.00,255.7,156.0,340.25
3,exp-period,ELITE,control,1st rr,63.20,15.07,15.38,131.5,111.0,184.75,202.0,110.0,266.00
5,exp-period,ELITE,viewed,1st rr,63.74,14.14,15.95,128.1,108.0,185.00,192.5,109.0,264.00
6,exp-period,GOLD,control,1st rr,57.82,18.30,15.12,133.5,113.0,188.00,231.2,138.0,325.00
8,exp-period,GOLD,viewed,1st rr,57.10,17.52,16.05,136.3,115.0,191.00,232.3,139.0,318.75
9,exp-period,HH,control,1st rr,56.82,15.83,16.01,140.0,119.5,190.25,329.1,205.0,445.00
11,exp-period,HH,viewed,1st rr,56.13,16.30,16.45,134.1,110.5,186.50,288.9,179.0,393.00
12,exp-period,INACTIVE,control,1st rr,49.23,21.48,17.44,134.3,114.0,193.50,244.7,131.0,353.50
14,exp-period,INACTIVE,viewed,1st rr,48.44,21.04,16.42,142.8,132.0,201.00,255.5,128.0,342.75


In [45]:
df4[['time_window',	'taxi_retention_segment',	'group_tc_flag', 
        'g2n %',	'cobra %',	'ocara %',
        'cobra_ttc_mean', 'cobra_ttc_p50', 'cobra_ttc_p75', 'ocara_ttc_mean', 'ocara_ttc_p50', 'ocara_ttc_p75']]

,time_window,taxi_retention_segment,group_tc_flag,g2n %,cobra %,ocara %,cobra_ttc_mean,cobra_ttc_p50,cobra_ttc_p75,ocara_ttc_mean,ocara_ttc_p50,ocara_ttc_p75
0,pre-period,DORMANT,control,0.05,43.50,33.83,124.9,105.0,178.00,234.6,124.0,318.00
1,pre-period,DORMANT,test,0.11,43.83,34.24,122.8,104.0,173.00,252.2,134.0,331.00
2,pre-period,ELITE,control,55.36,21.03,15.52,119.8,99.0,171.00,184.2,102.0,247.00
3,pre-period,ELITE,test,54.74,21.31,15.63,121.4,99.0,174.00,183.3,101.0,245.00
4,pre-period,GOLD,control,56.76,20.12,14.51,122.9,102.0,173.50,232.8,134.0,316.00
5,pre-period,GOLD,test,56.41,20.14,14.63,122.5,102.0,173.00,234.9,134.0,315.00
6,pre-period,HH,control,59.39,16.49,14.32,126.1,102.0,178.00,296.1,166.0,396.00
7,pre-period,HH,test,58.68,17.05,14.38,122.4,101.0,171.00,288.6,171.0,405.00
8,pre-period,INACTIVE,control,0.12,39.92,35.73,125.7,106.0,184.00,257.1,144.0,361.00
9,pre-period,INACTIVE,test,0.05,40.35,34.99,124.8,100.0,176.00,241.9,128.0,324.50


In [46]:
df_merge_exp_period

,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment,userid,group_tc_flag,pickup_distance_bucket,time_temporal,row_number,rr_rank
3143745,test,20250416,5737c6baddbec2203f7331d9,Link,67ff3acee888171d90fa7b30,dropped,NaN,1744779982040,1.744780e+12,NaN,NaN,NaN,NaN,1744779982040,1030,103622,4.841154,0.530,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,501M-1KM,1. morning,1,1st rr
3143746,test,20250419,5737c6baddbec2203f7331d9,Link,68033311f15bed13e145d6d2,dropped,NaN,1745040145019,1.745040e+12,NaN,NaN,NaN,NaN,1745040145019,1045,105225,3.040000,1.377,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,1.01-2KM,1. morning,2,2nd rr
739258,control,20250419,5737c6e3ddbec2203f733341,Link,6803553d39dca47f7e594927,dropped,NaN,1745048893140,1.745049e+12,NaN,NaN,NaN,NaN,1745048893140,1315,131813,2.700000,0.226,exp-period,DAILY,LINK_AUTO,ELITE,NaN,control,0-500M,2. afternoon,1,1st rr
739259,control,20250419,5737c6e3ddbec2203f733341,Link,68035ee38bd4380737cf7156,OCARA,1.745052e+12,1745051363720,1.745052e+12,0.905014,NaN,467.0,0.0,1745051363720,1345,135923,4.550000,0.686,exp-period,DAILY,LINK_AUTO,ELITE,NaN,control,501M-1KM,2. afternoon,2,2nd rr
739261,control,20250419,5737c6e3ddbec2203f733341,Link,680361bf2859a3306f4ed6a8,dropped,NaN,1745052095543,1.745052e+12,NaN,NaN,NaN,NaN,1745052095543,1400,141135,4.090000,0.031,exp-period,DAILY,LINK_AUTO,ELITE,NaN,control,0-500M,2. afternoon,3,2plus rr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015924,test,20250416,67f6a5f0316a328e07386f84,Auto,67ffbadc36ad326bd692806b,expired,NaN,1744812764693,NaN,NaN,NaN,NaN,NaN,1744812764693,1930,194244,3.780000,NaN,exp-period,UNKNOWN,UNKNOWN,HH,67f6a5f0316a328e07386f84,viewed,NA,3. evening,1,1st rr
2015926,test,20250416,67f6a5f0316a328e07386f84,Auto,67ffc344f5e56530911bf52e,dropped,NaN,1744814916212,1.744815e+12,NaN,NaN,NaN,NaN,1744814916212,2015,201836,3.520000,0.336,exp-period,UNKNOWN,UNKNOWN,HH,67f6a5f0316a328e07386f84,viewed,0-500M,3. evening,2,2nd rr
1407093,test,20250413,67f6aaced25c763a50d8bbef,Auto,67fb7af80f53d233192ed89a,dropped,NaN,1744534264196,1.744534e+12,NaN,NaN,NaN,NaN,1744534264196,1415,142104,2.300000,0.296,exp-period,UNKNOWN,UNKNOWN,HH,NaN,test,0-500M,2. afternoon,1,1st rr
176628,control,20250412,67f6af6dedbb784d97740f17,Auto,67f9de42d9a4a17e84349ca3,dropped,NaN,1744428610041,1.744429e+12,NaN,NaN,NaN,NaN,1744428610041,0900,090010,3.560000,2.590,exp-period,RARE_NEED,ONLY_AUTO,GOLD,NaN,control,2KM+,1. morning,1,1st rr


### FM Analysis

In [47]:
get_funnel(df_merge_exp_period[(df_merge_exp_period['rr_rank'] == '1st rr') & df_merge_exp_period['group_tc_flag'].isin(['control', 'viewed'])], ['time_window', 'pickup_distance_bucket', 'group_tc_flag','rr_rank'])

,time_window,pickup_distance_bucket,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,exp-period,0-500M,control,1st rr,25983,25983,22103,85.07,0.00,14.20,0.00,0.73,NaN,NaN,NaN,NaN,NaN,NaN,219.7,42.0,127.0,307.0,540.1,721.55,0,0,3690,171,19
1,exp-period,0-500M,viewed,1st rr,24221,24221,20677,85.37,0.00,14.00,0.00,0.63,NaN,NaN,NaN,NaN,NaN,NaN,207.6,39.0,123.0,291.0,497.9,657.45,0,0,3392,142,10
2,exp-period,1.01-2KM,control,1st rr,23280,23280,16903,72.61,0.00,25.28,0.00,2.11,NaN,NaN,NaN,NaN,NaN,NaN,243.3,53.0,137.0,326.0,596.6,769.60,0,0,5885,466,26
3,exp-period,1.01-2KM,viewed,1st rr,19837,19837,13904,70.09,0.00,27.51,0.00,2.39,NaN,NaN,NaN,NaN,NaN,NaN,238.9,50.0,135.0,319.0,576.6,772.00,0,0,5458,455,20
4,exp-period,2KM+,control,1st rr,8423,8423,5216,61.93,0.00,34.86,0.00,3.22,NaN,NaN,NaN,NaN,NaN,NaN,272.9,53.0,146.0,335.0,654.5,902.00,0,0,2936,252,19
5,exp-period,2KM+,viewed,1st rr,6408,6408,3810,59.46,0.00,37.06,0.00,3.48,NaN,NaN,NaN,NaN,NaN,NaN,249.0,54.0,134.0,305.5,605.6,829.20,0,0,2375,219,4
6,exp-period,501M-1KM,control,1st rr,25746,25746,20708,80.43,0.00,18.59,0.00,0.98,NaN,NaN,NaN,NaN,NaN,NaN,231.4,46.0,131.0,325.0,559.6,724.00,0,0,4785,232,21
7,exp-period,501M-1KM,viewed,1st rr,23591,23591,18918,80.19,0.00,18.72,0.00,1.09,NaN,NaN,NaN,NaN,NaN,NaN,221.1,44.0,126.0,302.0,530.4,713.20,0,0,4417,245,11
8,exp-period,NA,control,1st rr,28119,28119,0,0.00,69.30,0.00,1.73,28.96,133.3,59.0,113.0,188.0,267.0,307.0,NaN,NaN,NaN,NaN,NaN,NaN,486,19487,0,8143,0
9,exp-period,NA,viewed,1st rr,23357,23357,0,0.00,69.31,0.00,1.51,29.18,136.0,60.0,116.0,192.0,271.0,310.0,NaN,NaN,NaN,NaN,NaN,NaN,352,16188,0,6816,0


In [48]:
get_funnel(df_merge_exp_period[(df_merge_exp_period['rr_rank'] == '1st rr') 
                               & df_merge_exp_period['group_tc_flag'].isin(['control', 'viewed']) ], 
           ['time_window', 'taxi_regularity_segment', 'pickup_distance_bucket', 'group_tc_flag','rr_rank'])

,time_window,taxi_regularity_segment,pickup_distance_bucket,group_tc_flag,rr_rank,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,exp-period,BI_WEEKLY,0-500M,control,1st rr,6009,6009,5102,84.91,0.00,14.26,0.00,0.83,NaN,NaN,NaN,NaN,NaN,NaN,223.6,45.00,138.0,317.00,539.8,709.20,0,0,857,45,5
1,exp-period,BI_WEEKLY,0-500M,viewed,1st rr,5159,5159,4382,84.94,0.00,14.42,0.00,0.64,NaN,NaN,NaN,NaN,NaN,NaN,209.5,44.75,129.5,287.50,486.2,657.70,0,0,744,30,3
2,exp-period,BI_WEEKLY,1.01-2KM,control,1st rr,5592,5592,4075,72.87,0.00,24.77,0.00,2.36,NaN,NaN,NaN,NaN,NaN,NaN,245.4,54.00,139.0,368.00,609.0,763.80,0,0,1385,125,7
3,exp-period,BI_WEEKLY,1.01-2KM,viewed,1st rr,4234,4234,3005,70.97,0.00,26.29,0.00,2.74,NaN,NaN,NaN,NaN,NaN,NaN,241.0,58.00,154.0,326.00,574.8,741.80,0,0,1113,112,4
4,exp-period,BI_WEEKLY,2KM+,control,1st rr,1988,1988,1239,62.32,0.00,34.36,0.00,3.32,NaN,NaN,NaN,NaN,NaN,NaN,270.1,60.00,152.0,333.00,665.4,873.90,0,0,683,59,7
5,exp-period,BI_WEEKLY,2KM+,viewed,1st rr,1376,1376,815,59.23,0.00,36.34,0.00,4.43,NaN,NaN,NaN,NaN,NaN,NaN,270.5,53.75,137.0,320.50,599.1,911.00,0,0,500,61,0
6,exp-period,BI_WEEKLY,501M-1KM,control,1st rr,6150,6150,4979,80.96,0.00,18.00,0.00,1.04,NaN,NaN,NaN,NaN,NaN,NaN,227.6,46.00,128.0,311.50,568.0,734.40,0,0,1107,55,9
7,exp-period,BI_WEEKLY,501M-1KM,viewed,1st rr,4873,4873,3821,78.41,0.00,20.40,0.00,1.19,NaN,NaN,NaN,NaN,NaN,NaN,228.2,50.25,131.0,311.75,572.0,771.00,0,0,994,55,3
8,exp-period,BI_WEEKLY,NA,control,1st rr,6845,6845,0,0.00,70.80,0.00,1.87,27.30,133.4,61.0,112.0,186.00,265.0,310.75,NaN,NaN,NaN,NaN,NaN,NaN,128,4846,0,1869,0
9,exp-period,BI_WEEKLY,NA,viewed,1st rr,5229,5229,0,0.00,69.67,0.00,1.63,28.71,137.7,63.0,120.0,192.00,271.0,313.00,NaN,NaN,NaN,NaN,NaN,NaN,85,3643,0,1501,0


### Next 7 days RR check

In [49]:
def get_rr_retention(df):

    df['ride_date'] = pd.to_datetime(df['yyyymmdd'], format='%Y%m%d')
    
    df = df[(df['ride_date'] >= pd.Timestamp('2025-04-12')) & (df['ride_date'] <= pd.Timestamp('2025-04-26'))]
    
    first_rides = df[(df['ride_date'] >= pd.Timestamp('2025-04-12')) & (df['ride_date'] <= pd.Timestamp('2025-04-19'))]\
                    .sort_values(['customer_id', 'ride_date'])\
                    .drop_duplicates('customer_id', keep='first')\
                    .rename(columns={'ride_date': 'first_ride_date'})\
                    [['customer_id', 'first_ride_date']]
    
    df = df.merge(first_rides, on='customer_id', how='inner')
    

    df['days_since_first_ride'] = (df['ride_date'] - df['first_ride_date']).dt.days
    
    valid_orders = df[(df['days_since_first_ride'] >= 1) & (df['days_since_first_ride'] <= 7)]
    
    order_counts = valid_orders.groupby('customer_id')['order_id'].nunique().reset_index()
    order_counts.rename(columns={'order_id': 'orders_in_next_7_days'}, inplace=True)    
    
    return order_counts

def retention_calc():
    
    df1 = df_merge[df_merge['time_window'].isin(['exp-period'])][['customer_id', 'group_tc', 'group_tc_flag', 'taxi_regularity_segment']].drop_duplicates()
    
    df_rr_retention = get_rr_retention(df_merge[df_merge['time_window'].isin(['post-period', 'exp-period'])])

    df = pd.merge(df1,df_rr_retention, how='left', on='customer_id')

    return df

df_rr_window = retention_calc()

In [50]:
def get_retention_funnel(grouper):
    df = df_rr_window\
        .groupby(grouper)\
        .agg(customers = ('customer_id', 'nunique'),
             orders_in_next_7_days = ('orders_in_next_7_days', 'sum'))\
        .reset_index()
    df['rrpc_next_7days'] = (df['orders_in_next_7_days']/df['customers']).round(2)

    return df

In [51]:
get_retention_funnel(['group_tc'])

,group_tc,customers,orders_in_next_7_days,rrpc_next_7days
0,control,113463,364801.0,3.22
1,test,196046,629329.0,3.21


In [52]:
get_retention_funnel(['group_tc', 'group_tc_flag'])

,group_tc,group_tc_flag,customers,orders_in_next_7_days,rrpc_next_7days
0,control,control,113463,364801.0,3.22
1,test,test,96787,180521.0,1.87
2,test,viewed,99259,448808.0,4.52


In [53]:
get_retention_funnel(['taxi_regularity_segment', 'group_tc', 'group_tc_flag'])

,taxi_regularity_segment,group_tc,group_tc_flag,customers,orders_in_next_7_days,rrpc_next_7days
0,BI_WEEKLY,control,control,27072,54196.0,2.00
1,BI_WEEKLY,test,test,25528,36148.0,1.42
2,BI_WEEKLY,test,viewed,21262,57903.0,2.72
3,DAILY,control,control,16824,149872.0,8.91
4,DAILY,test,test,8353,48622.0,5.82
5,DAILY,test,viewed,21056,213130.0,10.12
6,MONTHLY,control,control,14614,21146.0,1.45
7,MONTHLY,test,test,15223,14966.0,0.98
8,MONTHLY,test,viewed,10479,21385.0,2.04
9,RARE_NEED,control,control,10037,15430.0,1.54


#### Pre Period 

In [252]:
def get_prev_7day(df):
    df['ride_date'] = pd.to_datetime(df['yyyymmdd'], format='%Y%m%d')
    
    df = df[(df['ride_date'] >= pd.Timestamp('2025-04-05')) & (df['ride_date'] <= pd.Timestamp('2025-04-26'))]
    
    
    first_rides = (
        df[(df['ride_date'] >= pd.Timestamp('2025-04-12')) & (df['ride_date'] <= pd.Timestamp('2025-04-19'))]
        .sort_values(['customer_id', 'ride_date'])
        .drop_duplicates('customer_id', keep='first')
        .rename(columns={'ride_date': 'first_ride_date'})
        [['customer_id', 'first_ride_date']]
    )
    
    df = df.merge(first_rides, on='customer_id', how='inner')
    
    df['days_before_first_ride'] = (df['first_ride_date'] - df['ride_date']).dt.days
    
    pre_period_orders = df[(df['days_before_first_ride'] >= 1) & (df['days_before_first_ride'] <= 7)]
    
    pre_order_counts = pre_period_orders.groupby('customer_id')['order_id'].nunique().reset_index()
    pre_order_counts.rename(columns={'order_id': 'orders_in_prev_7_days'}, inplace=True)
    
    return pre_order_counts

def prev_rr_calc():
    
    df1 = df_merge[df_merge['time_window'].isin(['exp-period'])][['customer_id', 'group_tc', 'group_tc_flag', 'taxi_regularity_segment']].drop_duplicates()
    
    df_rr_retention = get_prev_7day(df_merge[df_merge['time_window'].isin(['pre-period', 'exp-period'])])

    df = pd.merge(df1,df_rr_retention, how='left', on='customer_id')

    return df

df_prev_rr_window = prev_rr_calc()

In [253]:
def get_prev_funnel(grouper):
    df = df_prev_rr_window\
        .groupby(grouper)\
        .agg(customers = ('customer_id', 'nunique'),
             orders_in_prev_7_days = ('orders_in_prev_7_days', 'sum'))\
        .reset_index()
    df['rrpc_prev_7days'] = (df['orders_in_prev_7_days']/df['customers']).round(2)

    return df

In [254]:
get_prev_funnel(['group_tc'])

,group_tc,customers,orders_in_prev_7_days,rrpc_prev_7days
0,control,113463,270646.0,2.39
1,test,196046,467785.0,2.39


In [255]:
get_prev_funnel(['group_tc', 'group_tc_flag'])

,group_tc,group_tc_flag,customers,orders_in_prev_7_days,rrpc_prev_7days
0,control,control,113463,270646.0,2.39
1,test,test,96787,125893.0,1.30
2,test,viewed,99259,341892.0,3.44


In [256]:
get_prev_funnel(['taxi_regularity_segment', 'group_tc', 'group_tc_flag'])

,taxi_regularity_segment,group_tc,group_tc_flag,customers,orders_in_prev_7_days,rrpc_prev_7days
0,BI_WEEKLY,control,control,27072,34060.0,1.26
1,BI_WEEKLY,test,test,25528,21768.0,0.85
2,BI_WEEKLY,test,viewed,21262,34643.0,1.63
3,DAILY,control,control,16824,135395.0,8.05
4,DAILY,test,test,8353,43278.0,5.18
5,DAILY,test,viewed,21056,195021.0,9.26
6,MONTHLY,control,control,14614,11094.0,0.76
7,MONTHLY,test,test,15223,7599.0,0.50
8,MONTHLY,test,viewed,10479,11018.0,1.05
9,RARE_NEED,control,control,10037,7957.0,0.79


#### Comparison

In [257]:
display(get_retention_funnel(['taxi_regularity_segment', 'group_tc', 'group_tc_flag']))
display(get_prev_funnel(['taxi_regularity_segment', 'group_tc', 'group_tc_flag']))

,taxi_regularity_segment,group_tc,group_tc_flag,customers,orders_in_next_7_days,rrpc_next_7days
0,BI_WEEKLY,control,control,27072,54196.0,2.00
1,BI_WEEKLY,test,test,25528,36148.0,1.42
2,BI_WEEKLY,test,viewed,21262,57903.0,2.72
3,DAILY,control,control,16824,149872.0,8.91
4,DAILY,test,test,8353,48622.0,5.82
5,DAILY,test,viewed,21056,213130.0,10.12
6,MONTHLY,control,control,14614,21146.0,1.45
7,MONTHLY,test,test,15223,14966.0,0.98
8,MONTHLY,test,viewed,10479,21385.0,2.04
9,RARE_NEED,control,control,10037,15430.0,1.54


,taxi_regularity_segment,group_tc,group_tc_flag,customers,orders_in_prev_7_days,rrpc_prev_7days
0,BI_WEEKLY,control,control,27072,34060.0,1.26
1,BI_WEEKLY,test,test,25528,21768.0,0.85
2,BI_WEEKLY,test,viewed,21262,34643.0,1.63
3,DAILY,control,control,16824,135395.0,8.05
4,DAILY,test,test,8353,43278.0,5.18
5,DAILY,test,viewed,21056,195021.0,9.26
6,MONTHLY,control,control,14614,11094.0,0.76
7,MONTHLY,test,test,15223,7599.0,0.50
8,MONTHLY,test,viewed,10479,11018.0,1.05
9,RARE_NEED,control,control,10037,7957.0,0.79


In [156]:
df_exp = df_merge[(df_merge['time_window'].isin(['exp-period'])) & (df_merge['taxi_regularity_segment'] == 'DAILY') ]\
.groupby(['customer_id', 'group_tc_flag'])\
.agg(rr_count = ('order_id', 'nunique'))\
.sort_values(by=['rr_count'], ascending=False)\
.reset_index()

df_exp_list = df_exp['customer_id'].to_list()


df_per = df_merge[(df_merge['time_window'].isin(['pre-period'])) & (df_merge['taxi_regularity_segment'] == 'DAILY') ]\
.groupby(['customer_id', 'group_tc_flag'])\
.agg(rr_count = ('order_id', 'nunique'))\
.sort_values(by=['rr_count'], ascending=False)\
.reset_index()

In [157]:
df_exp_weekly = df_merge[(df_merge['time_window'].isin(['exp-period'])) & (df_merge['taxi_regularity_segment'] == 'WEEKLY') ]\
.groupby(['customer_id', 'group_tc_flag'])\
.agg(rr_count = ('order_id', 'nunique'))\
.sort_values(by=['rr_count'], ascending=False)\
.reset_index()

df_exp_weekly_list = df_exp_weekly['customer_id'].to_list()


df_per_weekly = df_merge[(df_merge['time_window'].isin(['pre-period'])) & (df_merge['taxi_regularity_segment'] == 'WEEKLY') ]\
.groupby(['customer_id', 'group_tc_flag'])\
.agg(rr_count = ('order_id', 'nunique'))\
.sort_values(by=['rr_count'], ascending=False)\
.reset_index()

In [158]:
print('Pre period')
print(df_per['rr_count'].quantile([i/10 for i in range(11)]))
print('experimental period')
print(df_exp['rr_count'].quantile([i/10 for i in range(11)]))

Pre period
0.0     1.0
0.1     2.0
0.2     4.0
0.3     5.0
0.4     6.0
0.5     8.0
0.6    10.0
0.7    12.0
0.8    14.0
0.9    19.0
1.0    91.0
Name: rr_count, dtype: float64
experimental period
0.0      1.0
0.1      2.0
0.2      3.0
0.3      5.0
0.4      6.0
0.5      8.0
0.6     10.0
0.7     12.0
0.8     16.0
0.9     21.0
1.0    142.0
Name: rr_count, dtype: float64


In [159]:
bins = [1, 4, 10, np.inf]
labels = ['1 to 4', '5 to 10', 'Above 10']

# Apply binning
df_per['rrpc_bucket'] = pd.cut(df_per['rr_count'], 
                                            bins=bins, 
                                            labels=labels,
                                            right=True,
                                            include_lowest=True)
df_exp['rrpc_bucket'] = pd.cut(df_exp['rr_count'], 
                                            bins=bins, 
                                            labels=labels,
                                            right=True,
                                            include_lowest=True)

# Apply binning
df_per_weekly['rrpc_bucket'] = pd.cut(df_per_weekly['rr_count'], 
                                            bins=bins, 
                                            labels=labels,
                                            right=True,
                                            include_lowest=True)
df_exp_weekly['rrpc_bucket'] = pd.cut(df_exp_weekly['rr_count'], 
                                            bins=bins, 
                                            labels=labels,
                                            right=True,
                                            include_lowest=True)

display(df_per.head(1))
display(df_exp.head(1))
display(df_per_weekly.head(1))
display(df_exp_weekly.head(1))

,customer_id,group_tc_flag,rr_count,rrpc_bucket
0,672d0531323c902419368122,control,91,Above 10


,customer_id,group_tc_flag,rr_count,rrpc_bucket
0,62711b2d20022e06e5851a90,viewed,142,Above 10


,customer_id,group_tc_flag,rr_count,rrpc_bucket
0,626648204e17b250f42eb091,control,73,Above 10


,customer_id,group_tc_flag,rr_count,rrpc_bucket
0,66b488cc1000550c95d07b90,viewed,89,Above 10


In [162]:
df_per_dist = df_per.groupby(['rrpc_bucket', 'group_tc_flag']).agg(customer_count = ('customer_id','nunique'), rr_count = ('rr_count', 'sum')).reset_index()

df_exp_dist = df_exp.groupby(['rrpc_bucket', 'group_tc_flag']).agg(customer_count = ('customer_id','nunique'),  rr_count = ('rr_count', 'sum')).reset_index()

In [164]:
df_per_dist['rrpc'] = df_per_dist['rr_count']/df_per_dist['customer_count']
df_exp_dist['rrpc'] = df_exp_dist['rr_count']/df_exp_dist['customer_count']

print('DAILY')
display(df_per_dist)
display(df_exp_dist)

DAILY


,rrpc_bucket,group_tc_flag,customer_count,rr_count,rrpc
0,1 to 4,control,4728,12024,2.543147
1,1 to 4,test,8128,20944,2.576772
2,5 to 10,control,6718,49184,7.321227
3,5 to 10,test,11740,85880,7.315162
4,Above 10,control,6119,105564,17.251839
5,Above 10,test,10746,185387,17.251722


,rrpc_bucket,group_tc_flag,customer_count,rr_count,rrpc
0,1 to 4,control,4676,11561,2.472412
1,1 to 4,test,4277,9707,2.269581
2,1 to 4,viewed,3922,10645,2.714176
3,5 to 10,control,5679,41324,7.276633
4,5 to 10,test,2687,18921,7.041682
5,5 to 10,viewed,7376,55009,7.457836
6,Above 10,control,6469,119060,18.404699
7,Above 10,test,1389,22814,16.424766
8,Above 10,viewed,9758,181623,18.612728


### 7 days RR Retention

In [54]:
def get_rr_retention(df):
    df['ride_date'] = pd.to_datetime(df['yyyymmdd'], format='%Y%m%d')
    
    df = df[(df['ride_date'] >= pd.Timestamp('2025-04-12')) & (df['ride_date'] <= pd.Timestamp('2025-04-26'))]
    
    first_rides = df[(df['ride_date'] >= pd.Timestamp('2025-04-12')) & (df['ride_date'] <= pd.Timestamp('2025-04-19'))]\
                    .sort_values(['customer_id', 'ride_date'])\
                    .drop_duplicates('customer_id', keep='first')\
                    .rename(columns={'ride_date': 'first_ride_date'})\
                    [['customer_id', 'first_ride_date']]
    
    df = df.merge(first_rides, on='customer_id', how='inner')
    
    df['days_since_first_ride'] = (df['ride_date'] - df['first_ride_date']).dt.days
    
    post7_rides = df[df['days_since_first_ride'] >= 8]

    retained_customers = post7_rides[['customer_id']].drop_duplicates()
    retained_customers['is_retained_post7'] = 1

    return retained_customers

In [55]:
def retention_calc():
    df1 = df_merge[df_merge['time_window'].isin(['exp-period'])][['customer_id', 'group_tc', 'group_tc_flag', 'taxi_regularity_segment']].drop_duplicates()
    
    df_rr_retention = get_rr_retention(df_merge[df_merge['time_window'].isin(['post-period', 'exp-period'])])
    
    df = pd.merge(df1, df_rr_retention, how='left', on='customer_id')
    df['is_retained_post7'] = df['is_retained_post7'].fillna(0).astype(int)

    return df

df_rr_window = retention_calc()

In [56]:
def get_retention_funnel(grouper):
    df = df_rr_window\
        .groupby(grouper)\
        .agg(customers=('customer_id', 'nunique'),
             retained=('is_retained_post7', 'sum'))\
        .reset_index()
    df['retention_rate_post7'] = (df['retained'] / df['customers']).round(2)

    return df


In [57]:
get_retention_funnel(['group_tc'])

,group_tc,customers,retained,retention_rate_post7
0,control,113463,46063,0.41
1,test,196046,79965,0.41


In [58]:
get_retention_funnel(['group_tc', 'group_tc_flag'])

,group_tc,group_tc_flag,customers,retained,retention_rate_post7
0,control,control,113463,46063,0.41
1,test,test,96787,28793,0.30
2,test,viewed,99259,51172,0.52


In [59]:
get_retention_funnel(['taxi_regularity_segment', 'group_tc', 'group_tc_flag'])

,taxi_regularity_segment,group_tc,group_tc_flag,customers,retained,retention_rate_post7
0,BI_WEEKLY,control,control,27072,8244,0.30
1,BI_WEEKLY,test,test,25528,6257,0.25
2,BI_WEEKLY,test,viewed,21262,8110,0.38
3,DAILY,control,control,16824,13352,0.79
4,DAILY,test,test,8353,5480,0.66
5,DAILY,test,viewed,21056,17767,0.84
6,MONTHLY,control,control,14614,3158,0.22
7,MONTHLY,test,test,15223,2693,0.18
8,MONTHLY,test,viewed,10479,2822,0.27
9,RARE_NEED,control,control,10037,2145,0.21


#### Prev period

In [60]:
def get_prev_7day(df):
    df['ride_date'] = pd.to_datetime(df['yyyymmdd'], format='%Y%m%d')
    
    df = df[(df['ride_date'] >= pd.Timestamp('2025-04-05')) & (df['ride_date'] <= pd.Timestamp('2025-04-26'))]

    first_rides = (
        df[(df['ride_date'] >= pd.Timestamp('2025-04-12')) & (df['ride_date'] <= pd.Timestamp('2025-04-19'))]
        .sort_values(['customer_id', 'ride_date'])
        .drop_duplicates('customer_id', keep='first')
        .rename(columns={'ride_date': 'first_ride_date'})
        [['customer_id', 'first_ride_date']]
    )
    
    df = df.merge(first_rides, on='customer_id', how='inner')
    
    df['days_before_first_ride'] = (df['first_ride_date'] - df['ride_date']).dt.days
    
    pre_period_riders = df[(df['days_before_first_ride'] >= 1) & (df['days_before_first_ride'] <= 7)]
    
    retained_customers = pre_period_riders[['customer_id']].drop_duplicates()
    retained_customers['is_retained_prev7'] = 1
    
    return retained_customers


In [61]:
def prev_rr_calc():
    df1 = df_merge[df_merge['time_window'].isin(['exp-period'])][['customer_id', 'group_tc', 'group_tc_flag', 'taxi_regularity_segment']].drop_duplicates()
    
    df_rr_retention = get_prev_7day(df_merge[df_merge['time_window'].isin(['pre-period', 'exp-period'])])

    df = pd.merge(df1, df_rr_retention, how='left', on='customer_id')
    df['is_retained_prev7'] = df['is_retained_prev7'].fillna(0).astype(int)

    return df
df_prev_rr_window = prev_rr_calc()

In [62]:
def get_prev_funnel(grouper):
    df = df_prev_rr_window\
        .groupby(grouper)\
        .agg(customers=('customer_id', 'nunique'),
             retained=('is_retained_prev7', 'sum'))\
        .reset_index()
    df['retention_rate_prev7'] = (df['retained'] / df['customers']).round(2)

    return df

In [63]:
get_prev_funnel(['group_tc'])

,group_tc,customers,retained,retention_rate_prev7
0,control,113463,52976,0.47
1,test,196046,90772,0.46


In [64]:
get_prev_funnel(['group_tc', 'group_tc_flag'])

,group_tc,group_tc_flag,customers,retained,retention_rate_prev7
0,control,control,113463,52976,0.47
1,test,test,96787,33827,0.35
2,test,viewed,99259,56945,0.57


In [65]:
get_prev_funnel(['taxi_regularity_segment', 'group_tc', 'group_tc_flag'])

,taxi_regularity_segment,group_tc,group_tc_flag,customers,retained,retention_rate_prev7
0,BI_WEEKLY,control,control,27072,9944,0.37
1,BI_WEEKLY,test,test,25528,7857,0.31
2,BI_WEEKLY,test,viewed,21262,9267,0.44
3,DAILY,control,control,16824,14541,0.86
4,DAILY,test,test,8353,6249,0.75
5,DAILY,test,viewed,21056,19145,0.91
6,MONTHLY,control,control,14614,3318,0.23
7,MONTHLY,test,test,15223,2708,0.18
8,MONTHLY,test,viewed,10479,2874,0.27
9,RARE_NEED,control,control,10037,1860,0.19


In [66]:
df_prev_rr_window[(df_prev_rr_window['taxi_regularity_segment'] == 'DAILY') 
                    & (df_prev_rr_window['is_retained_prev7'] > 0) 
                    & (df_prev_rr_window['group_tc_flag'] == 'viewed')]

,customer_id,group_tc,group_tc_flag,taxi_regularity_segment,is_retained_prev7
2,649badea21b6a53dc2451f14,test,viewed,DAILY,1
7,5cbfcedf54bc7263ff4a7db8,test,viewed,DAILY,1
17,5d41366f15e40f18eabf245f,test,viewed,DAILY,1
26,5e327e970bfd8c720616116c,test,viewed,DAILY,1
34,61a07e8df3de6fbd8a3a5271,test,viewed,DAILY,1
...,...,...,...,...,...
309463,63a132d7090adb88ca314d7e,test,viewed,DAILY,1
309467,5e520c4d02e760c9de99b79e,test,viewed,DAILY,1
309474,5ce84b4425ee3218d4ddc80f,test,viewed,DAILY,1
309483,65abdfabb0e4a127c24583f2,test,viewed,DAILY,1



### Action item

In [75]:
df_merge_exp_period.head(2)

,group_tc,yyyymmdd,customer_id,service_name,order_id,modified_order_status,customer_cancelled_epoch,order_requested_epoch,accepted_epoch,accept_to_cancelled,cobra_ttc,ocara_ttc,tta,epoch,quarter_hour,hhmmss,distance_final_distance,accept_to_pickup_distance,time_window,taxi_regularity_segment,taxi_service_affinity,taxi_retention_segment,userid,group_tc_flag,pickup_distance_bucket,time_temporal,row_number,rr_rank
3143745,test,20250416,5737c6baddbec2203f7331d9,Link,67ff3acee888171d90fa7b30,dropped,NaN,1744779982040,1.744780e+12,NaN,NaN,NaN,NaN,1744779982040,1030,103622,4.841154,0.530,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,501M-1KM,1. morning,1,1st rr
3143746,test,20250419,5737c6baddbec2203f7331d9,Link,68033311f15bed13e145d6d2,dropped,NaN,1745040145019,1.745040e+12,NaN,NaN,NaN,NaN,1745040145019,1045,105225,3.040000,1.377,exp-period,BI_WEEKLY,UNKNOWN,GOLD,NaN,test,1.01-2KM,1. morning,2,2nd rr


In [76]:
df_merge_exp_period.time_window.unique()

array(['exp-period'], dtype=object)

In [77]:
df_merge_exp_period.pickup_distance_bucket.unique()

array(['501M-1KM', '1.01-2KM', '0-500M', 'NA', '2KM+'], dtype=object)

##### Service level split
##### FM, all RRs

In [80]:
get_funnel(df_merge_exp_period, ['service_name', 'group_tc'])

,service_name,group_tc,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,Auto,control,84723,315086,164455,52.19,20.84,15.51,0.26,11.19,127.1,56.00,107.0,182.0,257.0,292.0,182.7,40.0,107.0,246.00,444.0,602.10,818,65660,48871,35196,77
1,Auto,test,146372,539017,282735,52.45,20.62,15.49,0.26,11.18,127.4,56.00,108.0,183.0,258.0,292.0,182.8,40.0,108.0,248.00,444.0,601.00,1393,111136,83500,60125,117
2,Bike Lite,control,4392,6437,3465,53.83,29.47,12.32,0.71,3.67,115.9,51.00,92.0,157.0,246.0,306.0,280.6,51.0,158.0,400.25,656.5,834.50,46,1897,793,229,7
3,Bike Lite,test,7343,10559,5847,55.37,28.04,12.45,0.50,3.63,120.0,53.00,99.0,163.0,250.0,295.9,270.3,58.0,181.5,381.00,621.0,810.40,53,2961,1315,361,22
4,CabEconomy,control,25605,55099,20421,37.06,33.56,15.93,2.36,11.09,156.2,64.00,123.0,223.0,332.0,390.0,285.7,52.0,158.0,376.00,678.0,909.20,1299,18490,8777,6075,37
5,CabEconomy,test,44777,95346,35255,36.98,33.34,15.89,2.34,11.45,157.5,65.00,126.0,224.0,334.0,391.0,281.8,54.0,159.0,376.00,668.0,911.00,2227,31790,15153,10872,47
6,CabPremium,control,3990,6920,3019,43.63,25.03,20.39,2.36,8.60,148.7,62.00,116.0,202.0,326.4,393.0,270.6,42.0,132.0,355.00,658.2,891.60,163,1732,1411,591,4
7,CabPremium,test,7074,12308,5259,42.73,25.09,21.50,2.27,8.40,151.2,60.25,116.0,213.0,324.0,394.0,267.3,36.0,124.0,327.00,651.0,869.00,280,3088,2646,1023,11
8,Link,control,43776,128311,62924,49.04,28.47,12.52,0.60,9.37,114.4,49.00,94.0,162.0,240.0,280.0,268.2,47.0,144.0,363.00,643.0,853.00,772,36526,16060,11852,177
9,Link,test,75011,218006,107559,49.34,28.40,12.60,0.56,9.10,115.2,49.00,95.0,165.0,240.0,281.0,255.0,44.0,135.0,345.00,623.0,821.15,1215,61915,27472,19561,279


In [81]:
get_funnel(df_merge_exp_period, ['service_name', 'group_tc', 'group_tc_flag'])

,service_name,group_tc,group_tc_flag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,Auto,control,control,84723,315086,164455,52.19,20.84,15.51,0.26,11.19,127.1,56.00,107.0,182.00,257.0,292.00,182.7,40.0,107.0,246.00,444.0,602.1,818,65660,48871,35196,77
1,Auto,test,test,55975,132229,68391,51.72,21.19,15.25,0.28,11.56,127.5,57.00,108.0,182.00,257.0,292.00,192.3,44.0,116.0,263.00,464.0,621.0,366,28017,20163,15254,32
2,Auto,test,viewed,90397,406788,214344,52.69,20.43,15.57,0.25,11.05,127.3,55.00,107.0,183.00,258.0,292.00,179.8,38.0,106.0,244.00,437.0,595.0,1027,83119,63337,44871,85
3,Bike Lite,control,control,4392,6437,3465,53.83,29.47,12.32,0.71,3.67,115.9,51.00,92.0,157.00,246.0,306.00,280.6,51.0,158.0,400.25,656.5,834.5,46,1897,793,229,7
4,Bike Lite,test,test,2639,3470,1902,54.81,28.53,12.28,0.46,3.92,120.3,55.00,99.0,166.00,251.8,295.00,274.0,61.5,197.0,394.50,654.0,834.3,16,990,426,127,9
5,Bike Lite,test,viewed,4704,7089,3945,55.65,27.80,12.54,0.52,3.48,119.8,52.25,98.0,162.00,250.0,295.30,268.5,58.0,171.0,373.00,607.6,798.6,37,1971,889,234,13
6,CabEconomy,control,control,25605,55099,20421,37.06,33.56,15.93,2.36,11.09,156.2,64.00,123.0,223.00,332.0,390.00,285.7,52.0,158.0,376.00,678.0,909.2,1299,18490,8777,6075,37
7,CabEconomy,test,test,15057,25874,9542,36.88,33.15,15.30,3.12,11.56,155.2,65.00,125.0,220.00,327.0,385.00,299.7,55.0,171.0,394.75,693.0,934.0,806,8576,3960,2973,17
8,CabEconomy,test,viewed,29720,69472,25713,37.01,33.41,16.11,2.05,11.41,158.3,64.00,127.0,225.00,337.0,394.00,275.8,54.0,157.0,371.00,659.9,903.9,1421,23214,11193,7899,30
9,CabPremium,control,control,3990,6920,3019,43.63,25.03,20.39,2.36,8.60,148.7,62.00,116.0,202.00,326.4,393.00,270.6,42.0,132.0,355.00,658.2,891.6,163,1732,1411,591,4


In [93]:
get_funnel(df_merge_exp_period, ['pickup_distance_bucket', 'group_tc', 'group_tc_flag'])

,pickup_distance_bucket,group_tc,group_tc_flag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,NA,control,control,56900,176594,0,0.00,70.39,0.00,1.75,27.85,127.8,55.0,105.0,181.0,262.0,302.0,NaN,NaN,NaN,NaN,NaN,NaN,3098,124305,0,49182,0
1,NA,test,test,40167,94559,0,0.00,70.51,0.00,1.97,27.51,127.5,56.0,106.0,181.0,259.0,299.0,NaN,NaN,NaN,NaN,NaN,NaN,1865,66672,0,26013,0
2,NA,test,viewed,57623,205128,0,0.00,70.31,0.00,1.61,28.08,129.0,54.0,106.0,184.0,265.0,305.0,NaN,NaN,NaN,NaN,NaN,NaN,3303,144218,0,57597,0
3,a. 0-500M,control,control,56214,104275,87953,84.35,0.00,15.01,0.00,0.65,NaN,NaN,NaN,NaN,NaN,NaN,197.0,36.0,110.5,269.0,483.0,646.00,0,0,15647,605,70
4,a. 0-500M,test,test,37581,52291,44113,84.36,0.00,14.87,0.00,0.77,NaN,NaN,NaN,NaN,NaN,NaN,219.7,41.0,125.0,296.0,516.0,681.00,0,0,7774,354,50
5,a. 0-500M,test,viewed,59376,126797,106981,84.37,0.00,15.02,0.00,0.60,NaN,NaN,NaN,NaN,NaN,NaN,186.3,34.0,105.0,262.0,461.0,613.00,0,0,19049,712,55
6,b. 501M-1KM,control,control,57225,103063,81382,78.96,0.00,20.03,0.00,1.01,NaN,NaN,NaN,NaN,NaN,NaN,206.4,40.0,114.0,287.0,515.0,677.35,0,0,20645,957,79
7,b. 501M-1KM,test,test,39344,54593,43588,79.84,0.00,19.04,0.00,1.12,NaN,NaN,NaN,NaN,NaN,NaN,222.7,43.0,126.0,314.0,543.0,720.00,0,0,10392,548,65
8,b. 501M-1KM,test,viewed,59318,123008,96816,78.71,0.00,20.25,0.00,1.05,NaN,NaN,NaN,NaN,NaN,NaN,198.2,37.0,111.0,274.0,495.0,652.15,0,0,24903,1227,62
9,c. 01-2KM,control,control,54387,94022,65268,69.42,0.00,28.25,0.00,2.33,NaN,NaN,NaN,NaN,NaN,NaN,219.6,47.0,122.0,285.0,529.7,713.00,0,0,26562,2088,104


In [94]:
get_funnel(df_merge_exp_period, ['pickup_distance_bucket', 'group_tc'])

,pickup_distance_bucket,group_tc,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,NA,control,56900,176594,0,0.00,70.39,0.00,1.75,27.85,127.8,55.0,105.0,181.0,262.0,302.0,NaN,NaN,NaN,NaN,NaN,NaN,3098,124305,0,49182,0
1,NA,test,97790,299687,0,0.00,70.37,0.00,1.72,27.90,128.6,55.0,106.0,183.0,263.0,303.0,NaN,NaN,NaN,NaN,NaN,NaN,5168,210890,0,83610,0
2,a. 0-500M,control,56214,104275,87953,84.35,0.00,15.01,0.00,0.65,NaN,NaN,NaN,NaN,NaN,NaN,197.0,36.0,110.5,269.0,483.0,646.00,0,0,15647,605,70
3,a. 0-500M,test,96957,179088,151094,84.37,0.00,14.98,0.00,0.65,NaN,NaN,NaN,NaN,NaN,NaN,195.7,35.0,110.0,272.0,475.0,636.00,0,0,26823,1066,105
4,b. 501M-1KM,control,57225,103063,81382,78.96,0.00,20.03,0.00,1.01,NaN,NaN,NaN,NaN,NaN,NaN,206.4,40.0,114.0,287.0,515.0,677.35,0,0,20645,957,79
5,b. 501M-1KM,test,98662,177601,140404,79.06,0.00,19.87,0.00,1.07,NaN,NaN,NaN,NaN,NaN,NaN,205.3,39.0,116.0,285.0,510.0,670.00,0,0,35295,1775,127
6,c. 01-2KM,control,54387,94022,65268,69.42,0.00,28.25,0.00,2.33,NaN,NaN,NaN,NaN,NaN,NaN,219.6,47.0,122.0,285.0,529.7,713.00,0,0,26562,2088,104
7,c. 01-2KM,test,93361,160743,111410,69.31,0.00,28.33,0.00,2.36,NaN,NaN,NaN,NaN,NaN,NaN,218.1,46.0,123.0,284.0,527.0,714.00,0,0,45538,3630,165
8,d. 2KM+,control,24799,33899,19681,58.06,0.00,38.52,0.00,3.42,NaN,NaN,NaN,NaN,NaN,NaN,243.1,47.0,126.0,293.0,590.0,835.50,0,0,13058,1111,49
9,d. 2KM+,test,42854,58117,33747,58.07,0.00,38.59,0.00,3.34,NaN,NaN,NaN,NaN,NaN,NaN,230.3,47.0,123.0,282.0,574.0,796.00,0,0,22430,1861,79


In [91]:
get_funnel(df_merge_exp_period, ['pickup_distance_bucket', 'service_name', 'group_tc'])

,pickup_distance_bucket,service_name,group_tc,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,NA,Auto,control,37117,97998,0,0.00,67.00,0.00,0.83,32.15,127.1,56.00,107.0,182.0,257.0,292.0,NaN,NaN,NaN,NaN,NaN,NaN,818,65660,0,31511,0
1,NA,Auto,test,63940,166080,0,0.00,66.92,0.00,0.84,32.24,127.4,56.00,108.0,183.0,258.0,292.0,NaN,NaN,NaN,NaN,NaN,NaN,1393,111136,0,53540,0
2,NA,Bike Lite,control,1429,2160,0,0.00,87.82,0.00,2.13,10.05,115.9,51.00,92.0,157.0,246.0,306.0,NaN,NaN,NaN,NaN,NaN,NaN,46,1897,0,217,0
3,NA,Bike Lite,test,2278,3336,0,0.00,88.76,0.00,1.59,9.65,120.0,53.00,99.0,163.0,250.0,295.9,NaN,NaN,NaN,NaN,NaN,NaN,53,2961,0,322,0
4,NA,CabEconomy,control,13921,25544,0,0.00,72.38,0.00,5.09,22.53,156.2,64.00,123.0,223.0,332.0,390.0,NaN,NaN,NaN,NaN,NaN,NaN,1299,18490,0,5755,0
5,NA,CabEconomy,test,24118,44339,0,0.00,71.70,0.00,5.02,23.28,157.5,65.00,126.0,224.0,334.0,391.0,NaN,NaN,NaN,NaN,NaN,NaN,2227,31790,0,10320,0
6,NA,CabPremium,control,1636,2449,0,0.00,70.72,0.00,6.66,22.62,148.7,62.00,116.0,202.0,326.4,393.0,NaN,NaN,NaN,NaN,NaN,NaN,163,1732,0,554,0
7,NA,CabPremium,test,2911,4333,0,0.00,71.27,0.00,6.46,22.25,151.2,60.25,116.0,213.0,324.0,394.0,NaN,NaN,NaN,NaN,NaN,NaN,280,3088,0,964,0
8,NA,Link,control,21306,48443,0,0.00,75.40,0.00,1.59,23.01,114.4,49.00,94.0,162.0,240.0,280.0,NaN,NaN,NaN,NaN,NaN,NaN,772,36526,0,11145,0
9,NA,Link,test,36409,81599,0,0.00,75.88,0.00,1.49,22.63,115.2,49.00,95.0,165.0,240.0,281.0,NaN,NaN,NaN,NaN,NaN,NaN,1215,61915,0,18464,0


In [92]:
get_funnel(df_merge_exp_period, ['pickup_distance_bucket', 'service_name', 'group_tc', 'group_tc_flag'])

,pickup_distance_bucket,service_name,group_tc,group_tc_flag,gross_customer,gross_orders,net_orders,g2n %,cobra %,ocara %,cobrn %,expired %,cobra_ttc_mean,cobra_ttc_p25,cobra_ttc_p50,cobra_ttc_p75,cobra_ttc_p90,cobra_ttc_p95,ocara_ttc_mean,ocara_ttc_p25,ocara_ttc_p50,ocara_ttc_p75,ocara_ttc_p90,ocara_ttc_p95,cobrm,cobra,ocara,expired,aborted
0,NA,Auto,control,control,37117,97998,0,0.00,67.00,0.00,0.83,32.15,127.1,56.00,107.0,182.00,257.0,292.00,NaN,NaN,NaN,NaN,NaN,NaN,818,65660,0,31511,0
1,NA,Auto,test,test,20106,41818,0,0.00,67.00,0.00,0.88,32.11,127.5,57.00,108.0,182.00,257.0,292.00,NaN,NaN,NaN,NaN,NaN,NaN,366,28017,0,13429,0
2,NA,Auto,test,viewed,43834,124262,0,0.00,66.89,0.00,0.83,32.28,127.3,55.00,107.0,183.00,258.0,292.00,NaN,NaN,NaN,NaN,NaN,NaN,1027,83119,0,40111,0
3,NA,Bike Lite,control,control,1429,2160,0,0.00,87.82,0.00,2.13,10.05,115.9,51.00,92.0,157.00,246.0,306.00,NaN,NaN,NaN,NaN,NaN,NaN,46,1897,0,217,0
4,NA,Bike Lite,test,test,802,1121,0,0.00,88.31,0.00,1.43,10.26,120.3,55.00,99.0,166.00,251.8,295.00,NaN,NaN,NaN,NaN,NaN,NaN,16,990,0,115,0
5,NA,Bike Lite,test,viewed,1476,2215,0,0.00,88.98,0.00,1.67,9.35,119.8,52.25,98.0,162.00,250.0,295.30,NaN,NaN,NaN,NaN,NaN,NaN,37,1971,0,207,0
6,NA,CabEconomy,control,control,13921,25544,0,0.00,72.38,0.00,5.09,22.53,156.2,64.00,123.0,223.00,332.0,390.00,NaN,NaN,NaN,NaN,NaN,NaN,1299,18490,0,5755,0
7,NA,CabEconomy,test,test,7434,12185,0,0.00,70.38,0.00,6.61,23.00,155.2,65.00,125.0,220.00,327.0,385.00,NaN,NaN,NaN,NaN,NaN,NaN,806,8576,0,2803,0
8,NA,CabEconomy,test,viewed,16684,32154,0,0.00,72.20,0.00,4.42,23.38,158.3,64.00,127.0,225.00,337.0,394.00,NaN,NaN,NaN,NaN,NaN,NaN,1421,23214,0,7517,0
9,NA,CabPremium,control,control,1636,2449,0,0.00,70.72,0.00,6.66,22.62,148.7,62.00,116.0,202.00,326.4,393.00,NaN,NaN,NaN,NaN,NaN,NaN,163,1732,0,554,0
